**Navigation** : [Index](README.md) | [<< Sudoku-17 Python](Sudoku-17-LLM-Python.ipynb) | [Sudoku-18 Python — source du portage](Sudoku-18-Comparison-Python.ipynb) | [Comparaison statistique (18b) >>](Sudoku-18b-Statistical-Comparison-Python.ipynb) | [Sudoku-19 (companion Lean) >>](Sudoku-19-Lean-Propagation-Lean.ipynb)

# Sudoku-Rust-Comparison : Comparaison des Solveurs de Sudoku (Rust)

Voir aussi : [Search Foundations](../Search/Part1-Foundations/) pour les algorithmes sous-jacents

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Classifier les différentes approches de résolution de Sudoku par catégorie (exacte, heuristique, propagation, apprentissage)
2. Comparer objectivement les performances de 4 solveurs Rust sur des puzzles de difficulté variee
3. Analyser les forces et faiblesses de chaque approche selon plusieurs critères (vitesse, fiabilite, scalabilite, simplicite)
4. Choisir le bon solveur en fonction du contexte d'utilisation

### Prerequis
- Avoir lu plusieurs carnets de la serie Sudoku (Backtracking, OR-Tools, Norvig, un autre au choix)
- Kernel Rust (evcxr) avec les crates `dlx`, `plotters`, `rand`, `rand_chacha` chargees par `:dep`

### Durée estimee : 35 minutes

## 1. Introduction

Au fil de cette serie de notebooks, nous avons explore **9 approches différentes** pour résoudre le Sudoku. Chacune incarne une philosophie algorithmique distincte :

| # | Notebook | Approche | Philosophie |
|---|----------|----------|-------------|
| 1 | [Backtracking](Sudoku-01-Backtracking-Python.ipynb) | Recherche exhaustive | Explorer systématiquement toutes les possibilites |
| 2 | [Genetic](Sudoku-03-Genetic-Python.ipynb) | Métaheuristique | Faire evoluer une population de solutions candidates |
| 3 | [OR-Tools](Sudoku-10-ORTools-Python.ipynb) | Programmation par contraintes | Declarer les contraintes, laisser le solveur chercher |
| 4 | [Z3](Sudoku-12-Z3-Python.ipynb) | Satisfiabilite (SMT) | Prouver l'existence d'une solution satisfaisant des formules logiques |
| 5 | [Dancing Links](Sudoku-02-DancingLinks-Python.ipynb) | Couverture exacte | Reduire a un problème de sélection de sous-ensembles |
| 6 | [Infer.NET](Sudoku-15-Infer-Python.ipynb) | Inference probabiliste | Propager des distributions de probabilité |
| 7 | [Norvig](Sudoku-07-Norvig-Python.ipynb) | Propagation de contraintes | Eliminer les impossibilites avant de chercher |
| 8 | Simulated Annealing | Recuit simule | Optimiser par perturbations aléatoires avec refroidissement |
| 10 | Neural Network | Apprentissage profond | Apprendre les motifs de résolution a partir d'exemples |

Ce notebook final propose une **comparaison systématique** de ces approches selon plusieurs dimensions : vitesse, fiabilite, elegance du code et scalabilite. Nous implementerons 4 solveurs Rust représentatifs (portage evcxr du carnet Python) et les testerons sur un benchmark commun.

In [2]:
:dep dlx = "0.8.5"
:dep plotters = { version = "0.3", features = ["evcxr", "evcxr_bitmap"] }
:dep rand = "0.8"
:dep rand_chacha = "0.3"

// Environnement : or-tools absent du siege (ORTOOLS_PREFIX/protoc non installes)
// -> solveur 4 en fallback DLX (crate dlx, Knuth TAoCP 7.2.2.1), desalignement
//    assume prevu par la spec #27665. Toutes les crates sont chargees ICI pour
//    eviter les recompilations evcxr intermediaires (qui perdent les variables
//    de types definis localement).
println!("dlx 0.8.5 (fallback CP-SAT), plotters 0.3 (evcxr), rand 0.8 + rand_chacha 0.3 charges.");
println!("Environnement charge.");

dlx 0.8.5 (fallback CP-SAT), plotters 0.3 (evcxr), rand 0.8 + rand_chacha 0.3 charges.


Environnement charge.


### Interpretation : Environnement et dependances

**Bibliotheques utilisees** :

| Bibliotheque | Version | Rôle dans ce notebook |
|-------------|---------|----------------------|
| dlx | 0.8.5 | Solveur de couverture exacte (dancing links, Knuth) — 4e solveur du benchmark |
| plotters | 0.3 (evcxr) | Visualisations (barres, boxplots) |
| rand / rand_chacha | 0.8 / 0.3 | Recuit simule reproductible (ChaCha8 seedee) |
| std::time | - | Mesure des temps d'exécution |

**Pourquoi ces choix** :
- **dlx** (dancing links) est la seule crate algorithmique externe : or-tools n'est pas installe sur ce siege (protoc absent), le 4e solveur utilise donc la crate `dlx` — désalignement documenté, prévu par la spec #27665. Les trois autrès solveurs sont implementes en Rust pur pour illustrer les algorithmes
- les statistiques du benchmark (medianne, percentiles) sont calculees en Rust pur (tri + selection)
- **plotters** génère les graphiques comparatifs (barres, boxplots)

> **Note** : Aucune dépendance lourde (pas de tensorflow, pytorch, etc.). Ce notebook reste léger et s'exécute sur n'importe quel kernel evcxr (Rust) standard.

## 2. Catégories d'approches

Les 9 approches etudiees se regroupent naturellement en **5 grandes catégories**, chacune correspondant a un paradigme algorithmique fondamental.

### 2.1 Méthodes exactes (garantie de solution)

Ces méthodes explorent l'espace de recherche de manière systématique et **trouvent toujours une solution** si elle existe.

| Méthode | Principe | Complexité pratique | Notebook |
|---------|----------|---------------------|----------|
| **Backtracking** | DFS + validation | Rapide (facile), lent (difficile) | [Sudoku-01](Sudoku-01-Backtracking-Python.ipynb) |
| **Backtracking + MRV** | DFS + heuristique MRV | Rapide même sur puzzles difficiles | [Sudoku-Python-BT](Sudoku-01-Backtracking-Python.ipynb) |
| **Dancing Links** | Couverture exacte (Algorithm X) | Optimal pour les problèmes de couverture | [Sudoku-02](Sudoku-02-DancingLinks-Python.ipynb) |
| **OR-Tools CP-SAT** | CP + SAT + CDCL | Très rapide, constant | [Sudoku-10](Sudoku-10-ORTools-Python.ipynb) |
| **Z3 SMT** | Satisfiabilite Modulo Théories | Rapide, overhead d'initialisation | [Sudoku-12](Sudoku-12-Z3-Python.ipynb) |

### 2.2 Méthodes de propagation

Ces méthodes **reduisent l'espace de recherche** en eliminant les valeurs impossibles avant (ou a la place de) la recherche.

| Méthode | Principe | Completude | Notebook |
|---------|----------|------------|----------|
| **Norvig** | Elimination + naked singles + DFS de secours | Complet (avec fallback recherche) | [Sudoku-07](Sudoku-07-Norvig-Python.ipynb) |
| **Stratégies humaines** | Règles logiques (naked pairs, X-Wing, etc.) | Incomplet sans recherche | (integre dans divers notebooks) |

### 2.3 Méthodes heuristiques (pas de garantie)

Ces méthodes utilisent des **mécanismes d'optimisation stochastique** et ne garantissent pas de trouver une solution.

| Méthode | Principe | Fiabilite | Notebook |
|---------|----------|-----------|----------|
| **Algorithme génétique** | Evolution de populations | Faible sur puzzles difficiles | [Sudoku-03](Sudoku-03-Genetic-Python.ipynb) |
| **Recuit simule** | Perturbations + refroidissement | Moyenne | Sudoku-04 |

### 2.4 Méthodes d'apprentissage

| Méthode | Principe | Fiabilite | Notebook |
|---------|----------|-----------|----------|
| **Réseau de neurones** | Apprentissage supervise | Approximation, necessite données | Sudoku-16 |

### 2.5 Méthodes probabilistes

| Méthode | Principe | Fiabilite | Notebook |
|---------|----------|-----------|----------|
| **Infer.NET** | Graphe de facteurs, inference bayesienne | Experimentale | [Sudoku-15](Sudoku-15-Infer-Python.ipynb) |

## 3. Implementations Rust des solveurs

Pour une comparaison equitable, nous implementons **4 solveurs représentatifs** dans le même environnement Rust (kernel evcxr). Chaque solveur représente une catégorie différente.

| Solveur | Catégorie | Pourquoi ce choix |
|---------|-----------|-------------------|
| **Backtracking simple** | Recherche exhaustive | Baseline naive, référence de comparaison |
| **Backtracking + MRV** | Recherche avec heuristique | Amelioration classique, montre l'impact des heuristiques |
| **Norvig (propagation)** | Propagation de contraintes | Approche elegante, souvent suffisante sans recherche |
| **DLX (dancing links)** | Couverture exacte | Algorithm X de Knuth (crate dlx), référence de performance |

### Struct SudokuGrid commune

Tous les solveurs utilisent la même représentation de grille pour garantir une comparaison equitable.

In [3]:
use std::time::Instant;

#[derive(Debug, Clone, PartialEq)]
enum SolverError {
    /// Levee par un solveur cooperatif quand son budget temps (deadline) est epuise.
    Timeout,
}

/// Representation d'une grille de Sudoku 9x9.
#[derive(Clone, PartialEq)]
struct SudokuGrid {
    cells: [[u8; 9]; 9],
}

impl SudokuGrid {
    fn new() -> Self {
        SudokuGrid { cells: [[0; 9]; 9] }
    }

    /// Cree une grille depuis une chaine de 81 caracteres (0 ou . = case vide).
    fn from_string(s: &str) -> Result<SudokuGrid, String> {
        let cleaned: String = s
            .chars()
            .filter(|c| !c.is_whitespace())
            .map(|c| if c == '.' { '0' } else { c })
            .collect();
        if cleaned.len() != 81 {
            return Err(format!("Attendu 81 caracteres, recu {}", cleaned.len()));
        }
        let mut grid = SudokuGrid::new();
        for (i, c) in cleaned.chars().enumerate() {
            grid.cells[i / 9][i % 9] = c as u8 - b'0';
        }
        Ok(grid)
    }

    /// Verifie si placer num a (row, col) respecte les contraintes.
    fn is_valid_placement(&self, row: usize, col: usize, num: u8) -> bool {
        if self.cells[row].contains(&num) {
            return false;
        }
        if (0..9).any(|r| self.cells[r][col] == num) {
            return false;
        }
        let (box_row, box_col) = (3 * (row / 3), 3 * (col / 3));
        for r in box_row..box_row + 3 {
            for c in box_col..box_col + 3 {
                if self.cells[r][c] == num {
                    return false;
                }
            }
        }
        true
    }

    /// Trouve la premiere case vide (parcours ligne par ligne).
    fn find_empty(&self) -> Option<(usize, usize)> {
        for r in 0..9 {
            for c in 0..9 {
                if self.cells[r][c] == 0 {
                    return Some((r, c));
                }
            }
        }
        None
    }

    fn count_empty(&self) -> usize {
        self.cells.iter().flatten().filter(|&&v| v == 0).count()
    }

    fn to_grid_string(&self) -> String {
        self.cells
            .iter()
            .flatten()
            .map(|&v| char::from_digit(v as u32, 10).unwrap())
            .collect()
    }
}

impl std::fmt::Display for SudokuGrid {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                writeln!(f, "{}", "-".repeat(21))?;
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                row_str.push(if val != 0 {
                    char::from_digit(val as u32, 10).unwrap()
                } else {
                    '.'
                });
                row_str.push(' ');
            }
            writeln!(f, "{}", row_str)?;
        }
        Ok(())
    }
}

// Verification rapide
let test = SudokuGrid::from_string("003020600900305001001806400008102900700000008006708200002609500800203009005010300")
    .expect("grille de test invalide");
println!("Grille de test:");
println!("{}", test);
println!("Cases vides: {}", test.count_empty());

Grille de test:


. . 3 | . 2 . | 6 . . 


9 . . | 3 . 5 | . . 1 


. . 1 | 8 . 6 | 4 . . 


---------------------


. . 8 | 1 . 2 | 9 . . 


... (3 outputs left out)

. . 2 | 6 . 9 | 5 . . 


8 . . | 2 . 3 | . . 9 


. . 5 | . 1 . | 3 . . 


Cases vides: 49


### Interpretation : Struct SudokuGrid - Représentation commune

**Fonctionnalites de la struct** :

| Méthode | Rôle | Utilite pour les solveurs |
|---------|------|--------------------------|
| `from_string` | Créé une grille depuis 81 caractères | Format universel pour les benchmarks |
| `clone` | Copie independante | Evite les interferences entre solveurs |
| `is_valid_placement` | Valide un placement (ligne/colonne/bloc) | Utilisee par le backtracking |
| `find_empty` | Trouve la première case vide | Parcours naive pour le backtracking simple |
| `count_empty` | Compte les cases vides | Métrique de difficulté |
| `to_string` | Convertit en chaîne | Interface avec le solveur Norvig |

**Grille de test** : 49 cases vides sur 81 (60% de la grille). C'est un puzzle de difficulté moyenne-facile qui servira de validation rapide pour chaque solveur.

**Design choisi** : La representation est volontairement simple (tableau [[u8; 9]; 9]) pour ne pas avantager un solveur particulier. Le solveur Norvig utilise sa propre représentation interne (tableau de candidats [u16; 81], un bit par chiffre) mais convertit vers/depuis SudokuGrid pour l'interface.

> **Point architecture** : Tous les solveurs implementent le même trait `SudokuSolver` (`fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError>`), ce qui permet de les comparer equitablement et de les utiliser de manière interchangeable dans le benchmark.

### 3.1 Solveur 1 : Backtracking simple

L'algorithme le plus intuitif : essayer chaque valeur dans chaque case vide, revenir en arriere si on atteint une impasse. C'est notre **baseline** -- le solveur le plus simple possible.

**Complexité** : O(9^m) dans le pire cas, ou m est le nombre de cases vides.

In [4]:
/// Solveur par backtracking simple (DFS sans heuristique).
struct BacktrackingSolver {
    call_count: usize,
    deadline: Option<Instant>, // annulation cooperative : echeance Instant
}

impl BacktrackingSolver {
    fn new() -> Self {
        BacktrackingSolver { call_count: 0, deadline: None }
    }

    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count = 0;
        self.backtrack(grid)
    }

    fn backtrack(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count += 1;
        // annulation cooperative : echeance consultee tous les 1024 appels
        if let Some(dl) = self.deadline {
            if (self.call_count & 1023) == 0 && Instant::now() > dl {
                return Err(SolverError::Timeout);
            }
        }
        let (row, col) = match grid.find_empty() {
            None => return Ok(true), // Toutes les cases remplies
            Some(rc) => rc,
        };
        for num in 1u8..10 {
            if grid.is_valid_placement(row, col, num) {
                grid.cells[row][col] = num;
                if self.backtrack(grid)? {
                    return Ok(true);
                }
                grid.cells[row][col] = 0; // Backtrack
            }
        }
        Ok(false)
    }
}

// Test rapide
let mut solver_bt = BacktrackingSolver::new();
let mut g = test.clone();
let start = Instant::now();
let solved = solver_bt.solve(&mut g).unwrap_or(false);
let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;
println!(
    "Backtracking simple: resolu={}, {} appels, {:.2} ms",
    solved, solver_bt.call_count, elapsed_ms
);

Backtracking simple: resolu=true, 201 appels, 0.03 ms


### Interpretation : Solveur Backtracking simple

**Résultat** : Le backtracking simple resout le puzzle de test en 201 appels recursifs (temps en ms en sortie live de la cellule ci-dessus -- règle #9434, non fige en prose car machine-dependant).

| Métrique | Valeur | Contexte |
|----------|--------|----------|
| Appels recursifs | 201 | Nombre d'entrees dans la boucle de backtracking |
| Temps | (ms live, règle #9434) | Rapide pour ce puzzle facile |
| Résultat | Succes | Solution trouvee |

**Analyse** :

1. **Pourquoi c'est rapide ici** : Le puzzle de test a 49 cases vides, c'est un puzzle facile. L'arbre de recherche est petit et le premier chemin explore mene souvent a la solution.

2. **Pourquoi ce sera lent sur les puzzles difficiles** : Sans heuristique, le solveur essaie les cases dans l'ordre (ligne par ligne, de gauche a droite) avec les valeurs 1 a 9. Sur un puzzle difficile avec 60+ cases vides, l'arbre de recherche peut atteindre des milliards de noeuds.

3. **Rôle de baseline** : Ce solveur sert de référence pour mesurer l'impact des améliorations. Toute optimisation (MRV, propagation, etc.) sera comparee a cette baseline.

> **Comparaison a venir** : Le solveur MRV suivant fera seulement 50 appels sur ce même puzzle -- déjà 4x moins. La différence sera beaucoup plus marquee sur les puzzles Medium et Hard.

## Exercice : Propagation des singletons (Naked Singles)

### Enonce

Le solveur Backtracking simple essaie les valeurs 1-9 séquentiellement. Mais avant de lancer le backtracking, on peut **propager** les deductions evidentes : les cases qui n'ont qu'un seul candidat possible.

Implementez `propagate_naked_singles(grid)` qui remplit recursivement toutes les cases ayant exactement 1 candidat, jusqu'a ce qu'aucune propagation ne soit possible.

> **Indices :**

- Étape 1 : Pour chaque case vide, calculez les candidats (absents de ligne + colonne + bloc)
- Étape 2 : Si exactement 1 candidat, remplissez la case
- Étape 3 : Repetez tant qu'au moins une case a ete remplie (propagation en cascade)
- Étape 4 : Retournez le nombre de cases remplies et la grille modifiée


In [5]:
// EXERCICE : Propagation des singletons (Naked Singles)
//
// Indications :
//   - Identifiez les cases avec exactement 1 candidat
//   - Remplissez-les et repetez (propagation en cascade)
//   - Retournez le nombre de cases remplies

fn propagate_naked_singles(_grid: &mut [[u8; 9]; 9]) -> usize {
    // Propage les naked singles jusqu'a stabilisation.
    //
    // TODO etudiant : implementez la propagation des naked singles
    0 // TODO
}

// Test sur un puzzle facile
let mut test_grid_18: [[u8; 9]; 9] = [
    [0,0,0,2,6,0,7,0,1],
    [6,8,0,0,7,0,0,9,0],
    [1,9,0,0,0,4,5,0,0],
    [8,2,0,1,0,0,0,4,0],
    [0,0,4,6,0,2,9,0,0],
    [0,5,0,0,0,3,0,2,8],
    [0,0,9,3,0,0,0,7,4],
    [0,4,0,0,5,0,0,3,6],
    [7,0,3,0,1,8,0,0,0],
];

let n_filled = propagate_naked_singles(&mut test_grid_18);
println!("Cases remplies par propagation : {}", n_filled);
for row in &test_grid_18 {
    println!("{:?}", row);
}

Cases remplies par propagation : 0


[0, 0, 0, 2, 6, 0, 7, 0, 1]


[6, 8, 0, 0, 7, 0, 0, 9, 0]


[1, 9, 0, 0, 0, 4, 5, 0, 0]


[8, 2, 0, 1, 0, 0, 0, 4, 0]


[0, 0, 4, 6, 0, 2, 9, 0, 0]


[0, 5, 0, 0, 0, 3, 0, 2, 8]


[0, 0, 9, 3, 0, 0, 0, 7, 4]


[0, 4, 0, 0, 5, 0, 0, 3, 6]


[7, 0, 3, 0, 1, 8, 0, 0, 0]


()

### 3.2 Solveur 2 : Backtracking avec MRV

L'heuristique **MRV** (Minimum Remaining Values) selectionne toujours la case ayant le **moins de valeurs possibles**. Cela detecte les impasses plus tot et réduit drastiquement l'arbre de recherche.

L'amélioration est souvent spectaculaire : de 100x a 1000x moins d'appels recursifs sur les puzzles difficiles.

In [6]:
/// Solveur par backtracking avec heuristique MRV (Minimum Remaining Values).
struct MRVBacktrackingSolver {
    call_count: usize,
    deadline: Option<Instant>,
}

impl MRVBacktrackingSolver {
    fn new() -> Self {
        MRVBacktrackingSolver { call_count: 0, deadline: None }
    }

    /// Retourne les valeurs possibles pour une case donnee (ordre croissant).
    fn get_possible(grid: &SudokuGrid, row: usize, col: usize) -> Vec<u8> {
        if grid.cells[row][col] != 0 {
            return vec![];
        }
        let mut possible: Vec<u8> = Vec::new();
        for num in 1u8..10 {
            if grid.is_valid_placement(row, col, num) {
                possible.push(num);
            }
        }
        possible
    }

    /// Trouve la case vide avec le moins de candidats.
    fn find_mrv(grid: &SudokuGrid) -> Option<(usize, usize, Vec<u8>)> {
        let mut best: Option<(usize, usize, Vec<u8>)> = None;
        let mut best_count = 10usize;
        for r in 0..9 {
            for c in 0..9 {
                if grid.cells[r][c] == 0 {
                    let possible = Self::get_possible(grid, r, c);
                    if possible.len() < best_count {
                        best_count = possible.len();
                        best = Some((r, c, possible));
                        if best_count == 0 {
                            return best; // Impasse detectee
                        }
                    }
                }
            }
        }
        best
    }

    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count = 0;
        self.backtrack(grid)
    }

    fn backtrack(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count += 1;
        if let Some(dl) = self.deadline {
            if (self.call_count & 1023) == 0 && Instant::now() > dl {
                return Err(SolverError::Timeout);
            }
        }
        let (row, col, possible) = match Self::find_mrv(grid) {
            None => return Ok(true),
            Some(rcp) => rcp,
        };
        if possible.is_empty() {
            return Ok(false);
        }
        for num in possible {
            grid.cells[row][col] = num;
            if self.backtrack(grid)? {
                return Ok(true);
            }
            grid.cells[row][col] = 0;
        }
        Ok(false)
    }
}

// Test rapide
let mut solver_mrv = MRVBacktrackingSolver::new();
let mut g = test.clone();
let start = Instant::now();
let solved = solver_mrv.solve(&mut g).unwrap_or(false);
let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;
println!(
    "Backtracking MRV: resolu={}, {} appels, {:.2} ms",
    solved, solver_mrv.call_count, elapsed_ms
);

Backtracking MRV: resolu=true, 50 appels, 0.38 ms


### Interpretation : Solveur Backtracking avec MRV

**Résultat** : Le backtracking avec MRV resout le puzzle en 50 appels recursifs (temps en ms en sortie live -- règle #9434).

| Métrique | Backtracking simple | BT + MRV | Amelioration |
|----------|--------------------|---------|-------------|
| Appels recursifs | 201 | 50 | **4x moins** |
| Temps | (ms live -- règle #9434) | (ms live) | Plus lent |

**Paradoxe apparent** : MRV fait moins d'appels mais prend plus de temps. Cela s'explique par le coût de l'heuristique :

- **Backtracking simple** : Chaque appel est très rapide (trouver la première case vide, essayer 1-9)
- **BT + MRV** : Chaque appel necessite de calculer les candidats de toutes les cases vides et de trouver celle avec le minimum

**Pourquoi MRV est quand même préférable** :
1. Sur les puzzles faciles, la différence de temps est negligeable (millisecondes)
2. Sur les puzzles difficiles, MRV réduit l'arbre de recherche de plusieurs ordres de grandeur
3. Le surcout par appel est constant, mais le nombre d'appels evites croit exponentiellement avec la difficulté

> **Principe MRV** : Toujours choisir la variable la plus contrainte ("fail-first"). En traitant d'abord les cases avec peu de candidats, on detecte les impasses plus tot et on evite d'explorer des sous-arbres inutiles.

### 3.3 Solveur 3 : Propagation de contraintes (Norvig)

L'approche de Peter Norvig combine deux stratégies de propagation :

1. **Elimination** : quand une case recoit une valeur, on retire cette valeur des candidats de tous ses voisins (même ligne, colonne, bloc)
2. **Naked single** : quand une valeur n'a qu'un seul emplacement possible dans une unite, on l'y assigne

Si la propagation seule ne suffit pas, un **backtracking avec MRV** prend le relais. En pratique, la propagation resout la majorite des puzzles faciles/moyens sans aucune recherche.

In [7]:
/// Solveur par propagation de contraintes (style Peter Norvig).
/// Candidats en masques binaires : bit (d-1) = chiffre d possible.
/// Reference : http://norvig.com/sudoku.html
const FULL_CANDIDATES: u16 = 0b1_1111_1111;
const ALL_DIGITS: u8 = 9;

fn cand_bit(d: u8) -> u16 {
    1u16 << (d - 1)
}

struct NorvigSolver {
    call_count: usize,
    deadline: Option<Instant>,
    /// 27 unites : 9 lignes, 9 colonnes, 9 blocs (indices de cases).
    units: Vec<Vec<usize>>,
    /// Pour chaque case, les indices de ses 3 unites.
    units_of: [[usize; 3]; 81],
    /// Pour chaque case, ses 20 voisins (pairs ligne/colonne/bloc).
    peers: Vec<Vec<usize>>,
}

impl NorvigSolver {
    fn new() -> Self {
        let mut units: Vec<Vec<usize>> = Vec::new();
        for r in 0..9 {
            units.push((0..81).filter(|&i| i / 9 == r).collect());
        }
        for c in 0..9 {
            units.push((0..81).filter(|&i| i % 9 == c).collect());
        }
        for br in 0..3 {
            for bc in 0..3 {
                units.push(
                    (0..81)
                        .filter(|&i| (i / 9) / 3 == br && (i % 9) / 3 == bc)
                        .collect(),
                );
            }
        }
        let mut units_of = [[0usize; 3]; 81];
        for s in 0..81 {
            let mut k = 0;
            for (u_idx, u) in units.iter().enumerate() {
                if u.contains(&s) {
                    units_of[s][k] = u_idx;
                    k += 1;
                }
            }
        }
        let mut peers: Vec<Vec<usize>> = Vec::with_capacity(81);
        for s in 0..81 {
            let mut set: Vec<usize> = Vec::new();
            for &u in &units_of[s] {
                for &s2 in &units[u] {
                    if s2 != s && !set.contains(&s2) {
                        set.push(s2);
                    }
                }
            }
            peers.push(set);
        }
        NorvigSolver { call_count: 0, deadline: None, units, units_of, peers }
    }

    /// Assigne d a s en eliminant toutes les autres valeurs.
    fn assign(&self, values: &mut [u16; 81], s: usize, d: u8) -> Result<(), ()> {
        let others = values[s] & !cand_bit(d);
        for d2 in 1..=ALL_DIGITS {
            if others & cand_bit(d2) != 0 {
                self.eliminate(values, s, d2)?;
            }
        }
        Ok(())
    }

    /// Elimine d des candidats de s et propage.
    fn eliminate(&self, values: &mut [u16; 81], s: usize, d: u8) -> Result<(), ()> {
        if values[s] & cand_bit(d) == 0 {
            return Ok(()); // Deja eliminee
        }
        values[s] &= !cand_bit(d);

        // Si aucun candidat restant : contradiction
        if values[s] == 0 {
            return Err(());
        }

        // Si un seul candidat : propager aux voisins
        if values[s].count_ones() == 1 {
            let d2 = values[s].trailing_zeros() as u8 + 1;
            for &s2 in &self.peers[s] {
                self.eliminate(values, s2, d2)?;
            }
        }

        // Si d n'a qu'un seul emplacement dans une unite : l'y assigner
        for &u in &self.units_of[s] {
            let mut dplaces: Vec<usize> = Vec::new();
            for &s2 in &self.units[u] {
                if values[s2] & cand_bit(d) != 0 {
                    dplaces.push(s2);
                }
            }
            if dplaces.is_empty() {
                return Err(()); // Contradiction
            }
            if dplaces.len() == 1 {
                self.assign(values, dplaces[0], d)?;
            }
        }
        Ok(())
    }

    /// Recherche DFS avec heuristique MRV.
    fn search(&mut self, values: &mut [u16; 81]) -> Result<Option<[u16; 81]>, SolverError> {
        self.call_count += 1;
        if let Some(dl) = self.deadline {
            if (self.call_count & 255) == 0 && Instant::now() > dl {
                return Err(SolverError::Timeout);
            }
        }

        // Verifie si resolu
        if values.iter().all(|&v| v.count_ones() == 1) {
            return Ok(Some(*values));
        }

        // MRV : case non resolue avec le moins de candidats (premier indice en cas d'egalite)
        let mut s_min = 81usize;
        let mut best_len = 10u32;
        for s in 0..81 {
            let len = values[s].count_ones();
            if len > 1 && len < best_len {
                best_len = len;
                s_min = s;
            }
        }

        for d in 1..=ALL_DIGITS {
            if values[s_min] & cand_bit(d) != 0 {
                let mut new_values = *values; // values.copy()
                if self.assign(&mut new_values, s_min, d).is_ok() {
                    if let Some(result) = self.search(&mut new_values)? {
                        return Ok(Some(result));
                    }
                }
            }
        }
        Ok(None)
    }

    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count = 0;
        let grid_str = grid.to_grid_string();
        let mut values = [FULL_CANDIDATES; 81];
        for (s, b) in grid_str.bytes().enumerate() {
            let d = b - b'0';
            if (1..=9).contains(&d) {
                if self.assign(&mut values, s, d).is_err() {
                    return Ok(false);
                }
            }
        }
        match self.search(&mut values)? {
            Some(result) => {
                for i in 0..81 {
                    grid.cells[i / 9][i % 9] = result[i].trailing_zeros() as u8 + 1;
                }
                Ok(true)
            }
            None => Ok(false),
        }
    }
}

// Test rapide
let mut solver_norvig = NorvigSolver::new();
let mut g = test.clone();
let start = Instant::now();
let solved = solver_norvig.solve(&mut g).unwrap_or(false);
let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;
println!(
    "Norvig: resolu={}, {} appels, {:.2} ms",
    solved, solver_norvig.call_count, elapsed_ms
);

Norvig: resolu=true, 1 appels, 0.58 ms


### Interpretation : Solveur Norvig (propagation de contraintes)

**Résultat** : Norvig resout le puzzle en **1 appel récursif** (temps en ms en sortie live -- règle #9434).

| Métrique | Valeur | Observation |
|----------|--------|-------------|
| Appels recursifs | 1 | Aucune recherche nécessaire |
| Temps | (ms live, règle #9434) | Le plus rapide sur ce puzzle |

**Pourquoi un seul appel ?**

La propagation de contraintes (elimination + naked singles) a entierement résolu le puzzle sans qu'aucun backtracking ne soit nécessaire. C'est le cas pour la plupart des puzzles faciles/moyens.

**Mécanismes de propagation** :
1. **Elimination** : Quand une valeur est assignee a une case, elle est retiree des candidats de tous ses voisins (ligne, colonne, bloc)
2. **Naked single** : Si une valeur ne peut aller que dans une seule case d'une unite, on l'y assigne
3. **Cascade** : Chaque assignation declenche une nouvelle propagation, qui peut elle-même declencher d'autrès assignations

**Comparaison avec le backtracking** : Le solveur simple a necessite 201 appels recursifs pour le même puzzle. Norvig atteint la même solution en un seul appel grâce a la propagation qui elimine les branches inutiles avant même de les explorer.

> **Point cle** : La force de Norvig n'est pas la vitesse pure, mais la capacité a résoudre sans recherche. Sur des puzzles plus difficiles, le backtracking MRV prend le relais avec un espace de recherche déjà drastiquement réduit.

> **Note sur les mesures de temps** : les temps de résolution varient selon la machine (ordonnancement CPU, cache). Les **nombres d'appels récursifs** et **coupes FC** sont déterministes (reproductibles à l'identique) ; seuls les temps en millisecondes sont indicatifs (ordre de grandeur).


### 3.4 Solveur 4 : DLX (dancing links)

OR-Tools CP-SAT n'est pas disponible sur ce siege Rust (protoc absent). Le 4e solveur utilise la crate **dlx** -- l'algorithme *dancing links* de Knuth (TAoCP 7.2.2.1), deja rencontre dans [Sudoku-02 Dancing Links](Sudoku-02-DancingLinks-Python.ipynb).

Au lieu de programmer un algorithme de recherche, on **traduit** le Sudoku en un probleme de couverture exacte :
- 324 contraintes (81 case-remplie, 81 ligne-chiffre, 81 colonne-chiffre, 81 bloc-chiffre)
- 729 options (une par paire case x chiffre)

Le solveur applique l'Algorithm X avec heuristique MRV sur cette matrice creuse. C'est le desalignement assume du portage (spec #27665) : meme famille << couverture exacte >>, moteur different de CP-SAT.

In [8]:
use dlx::x::{Problem as XProblem, make_problem as x_make_problem};
use dlx::{OptOrder, Solver as DlxSearch, Solve as _, Uint};
use dlx::choose::{mrv_chooser, prefer_any, no_tiebreak};

/// Solveur exact-cover (Dancing Links, Knuth TAoCP 7.2.2.1).
/// Remplace OR-Tools CP-SAT : or-tools absent du siege, fallback DLX crate
/// prevu par la spec #27665 (desalignement assume).
struct DlxSolver {
    call_count: usize, // non applicable, mais conserve pour l'interface
    deadline: Option<Instant>,
}

impl DlxSolver {
    fn new() -> Self {
        DlxSolver { call_count: 0, deadline: None }
    }

    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.call_count = 1; // Le solveur fait tout en interne
        // annulation cooperative : budget natif CP-SAT -> pre-verification ici
        if let Some(dl) = self.deadline {
            if Instant::now() > dl {
                return Err(SolverError::Timeout);
            }
        }
        let s: Vec<usize> = grid
            .to_grid_string()
            .bytes()
            .map(|b| (b - b'0') as usize)
            .collect();
        let box_no = |j: usize, k: usize| (j / 3) * 3 + (k / 3);

        // Items non satisfaits par les indices : case / ligne-chiffre / colonne-chiffre / bloc-chiffre
        let mut names: Vec<Uint> = Vec::new();
        for j in 0..9 {
            for k in 0..9 {
                if s[j * 9 + k] == 0 {
                    names.push(Uint(j * 9 + k));
                }
                for d in 0..9 {
                    if !(0..9).any(|c| s[j * 9 + c] == d + 1) {
                        names.push(Uint(81 + j * 9 + d));
                    }
                    if !(0..9).any(|r| s[r * 9 + k] == d + 1) {
                        names.push(Uint(162 + k * 9 + d));
                    }
                    if !(0..81).any(|i| box_no(i / 9, i % 9) == box_no(j, k) && s[i] == d + 1) {
                        names.push(Uint(243 + box_no(j, k) * 9 + d));
                    }
                }
            }
        }
        names.sort_unstable();
        names.dedup();
        let index_of = |id: Uint| -> Uint {
            names.binary_search(&id).unwrap() as Uint
        };

        // Options : (j, k, d) compatible avec tous les indices poses
        let mut os: Vec<Vec<Uint>> = Vec::new();
        for j in 0..9 {
            for k in 0..9 {
                for d in 0..9usize {
                    let row_ok = !(0..9).any(|c| s[j * 9 + c] == d + 1);
                    let col_ok = !(0..9).any(|r| s[r * 9 + k] == d + 1);
                    let box_ok = !(0..81)
                        .any(|i| box_no(i / 9, i % 9) == box_no(j, k) && s[i] == d + 1);
                    if s[j * 9 + k] == 0 && row_ok && col_ok && box_ok {
                        os.push(vec![
                            index_of(Uint(j * 9 + k)),
                            index_of(Uint(81 + j * 9 + d)),
                            index_of(Uint(162 + k * 9 + d)),
                            index_of(Uint(243 + box_no(j, k) * 9 + d)),
                        ]);
                    }
                }
            }
        }

        let mut problem: XProblem = x_make_problem(Uint(names.len()), 0, &os, OptOrder::Seq);
        let mut chooser = mrv_chooser(prefer_any(), no_tiebreak());
        let mut solver = DlxSearch::new(&mut problem);
        if solver.next_solution(&mut chooser) {
            let solution = solver.fmt_solution().to_vec();
            for i in solution {
                let opt = &os[i as usize];
                let j = (names[opt[0] as usize] / 9) as usize;
                let k = (names[opt[0] as usize] % 9) as usize;
                let d = ((names[opt[1] as usize] - 81) % 9) as usize + 1;
                grid.cells[j][k] = d as u8;
            }
            Ok(true)
        } else {
            Ok(false)
        }
    }
}

// Test rapide
let mut solver_dlx = DlxSolver::new();
let mut g = test.clone();
let start = Instant::now();
let solved = solver_dlx.solve(&mut g).unwrap_or(false);
let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;
println!("DLX (remplace CP-SAT): resolu={}, {:.2} ms", solved, elapsed_ms);

DLX (remplace CP-SAT): resolu=true, 0.86 ms


### Interpretation : Solveur DLX (dancing links)

**Resultat** : DLX resout le puzzle avec succes (temps en ms en sortie live -- regle #9434).

| Metrique | Valeur | Observation |
|----------|--------|-------------|
| Temps | (ms live, regle #9434) | Plus lent que Norvig sur ce puzzle (construction du probleme de couverture) |
| Appels | 1 | Le solveur gere tout en interne |

**Analyse** :

1. **Overhead de construction** : Le temps inclut la traduction du puzzle en probleme de couverture exacte (729 options x 324 contraintes) et l'initialisation. Pour un seul puzzle, cet overhead est significatif.

2. **Avantage sur les grands problemes** : L'Algorithm X de Knuth est concu pour les matrices creuses de grande taille. Le temps de resolution reste quasi-constant quelle que soit la difficulte du puzzle.

3. **Declaratif vs imperatif** : Le code ne contient **aucune logique de recherche** -- on construit le probleme et le solveur trouve la solution.

> **Quand utiliser DLX** : Quand la robustesse et la previsibilite sont plus importantes que la vitesse brute, ou quand le probleme se reduit naturellement a une couverture exacte.

## Exercice : Vérifier l'unicite de la solution avec le solveur exact

### Enonce

Un Sudoku bien pose possede exactement **une** solution. Avec la crate `dlx`, on peut le vérifier en énumérant les solutions une par une (méthode `next_solution` itérative). Implementez `has_unique_solution(grid)` qui retourne True si le puzzle a exactement 1 solution.

> **Indices :**

- Étape 1 : Construisez le problème de couverture exacte comme dans `DlxSolver` (clues -> make_problem)
- Étape 2 : Énumérez une première solution avec `next_solution`
- Étape 3 : Si une solution est trouvée, énumérez la suivante
- Étape 4 : Une deuxième solution existe = le puzzle n'est pas unique


In [9]:
// EXERCICE : Verifier l'unicite de la solution avec le solveur exact
//
// Indications :
//   - Resoudre une premiere fois
//   - Ajouter une contrainte interdisant cette solution
//   - Tenter de resoudre a nouveau
//
// NOTE (portage Rust) : OR-Tools CP-SAT est remplace par la crate dlx
// (fallback pre-autorise par la spec #27665) ; son interface iterative
// (next_solution) permet d'enumerer les solutions une par une.

fn has_unique_solution(_grid_str: &str) -> Option<bool> {
    // Verifie si un Sudoku a exactement une solution.
    //
    // Args: grille en chaine de 81 caracteres ('0' ou '.' = case vide)
    //
    // Returns:
    //   Some(true) si exactement 1 solution, Some(false) sinon
    //
    // TODO etudiant : implementez la verification d'unicite
    println!("Exercice a completer : has_unique_solution");
    None // TODO etudiant
}

// Test (dlx charge en cellule 2 : le solveur exact est disponible)
let easy = "000260701680070090190004500820100040004602900050003028009300074040050036703018000";
let unique = has_unique_solution(easy);
println!("Solution unique : {:?}", unique);

Exercice a completer : has_unique_solution


Solution unique : None


## 4. Benchmark

Nous testons les 4 solveurs sur **4 niveaux de difficulté** : 10 puzzles disponibles pour Easy/Medium/Hard (le benchmark en exécute **5 par difficulté**), et 1 puzzle Expert (régime 17-24 indices, où un seul suffit à illustrer les timeout). Tous les puzzles sont codes en dur ci-dessous pour garantir la reproductibilite (pas de dépendance a des fichiers externes).

### Jeux de test

| Difficulte | Caractéristiques | Nombre de cases vides |
|------------|------------------|-----------------------|
| **Easy** | Beaucoup d'indices, propagation simple suffit | ~35-45 |
| **Medium** | Indices reduits, necessite un peu de recherche | ~45-55 |
| **Hard** | Très peu d'indices, cas extrêmes connus | ~55-60 |
| **Expert** | Regime 17-24 indices : les solveurs exacts (backtracking) timeout, les métaheuristiques (recuit simule, génétique) n'ont aucune garantie de convergence | ~57-64 |

In [10]:
// -- Jeux de test codes en dur --
// Source : Sudoku_Easy51.txt (10 premiers)
const EASY_PUZZLES: [&str; 10] = [
    "003020600900305001001806400008102900700000008006708200002609500800203009005010300",
    "200080300060070084030500209000105408000000000402706000301007040720040060004010003",
    "000000907000420180000705026100904000050000040000507009920108000034059000507000000",
    "030050040008010500460000012070502080000603000040109030250000098001020600080060020",
    "020810740700003100090002805009040087400208003160030200302700060005600008076051090",
    "100920000524010000000000070050008102000000000402700090060000000000030945000071006",
    "043080250600000000000001094900004070000608000010200003820500000000000005034090710",
    "480006902002008001900370060840010200003704100001060049020085007700900600609200018",
    "000900002050123400030000160908000000070000090000000205091000050007439020400007000",
    "001900003900700160030005007050000009004302600200000070600100030042007006500006800",
];

// Source : Sudoku_top95.txt (10 puzzles selectionnes, difficulte medium)
const MEDIUM_PUZZLES: [&str; 10] = [
    "4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......",
    "52...6.........7.13...........4..8..6......5...........418.........3..2...87.....",
    "6.....8.3.4.7.................5.4.7.3..2.....1.6.......2.....5.....8.6......1....",
    "48.3............71.2.......7.5....6....2..8.............1.76...3.....4......5....",
    "....14....3....2...7..........9...3.6.1.............8.2.....1.4....5.6.....7.8...",
    "......52..8.4......3...9...5.1...6..2..7........3.....6...1..........7.4.......3.",
    "6.2.5.........3.4..........43...8....1....2........7..5..27...........81...6.....",
    ".524.........7.1..............8.2...3.....6...9.5.....1.6.3...........897........",
    "6.2.5.........4.3..........43...8....1....2........7..5..27...........81...6.....",
    ".923.........8.1...........1.7.4...........658.........6.5.2...4.....7.....9.....",
];

// Source : Sudoku_hardest.txt (les 10 premiers relevent du regime "Hard" :
// backtracking galere, propagation suffit).
const HARD_PUZZLES: [&str; 10] = [
    "85...24..72......9..4.........1.7..23.5...9...4...........8..7..17..........36.4.",
    "..53.....8......2..7..1.5..4....53...1..7...6..32...8..6.5....9..4....3......97..",
    "12..4......5.69.1...9...5.........7.7...52.9..3......2.9.6...5.4..9..8.1..3...9.4",
    "...57..3.1......2.7...234......8...4..7..4...49....6.5.42...3.....7..9....18.....",
    "7..1523........92....3.....1....47.8.......6............9...5.6.4.9.7...8....6.1.",
    "1....7.9..3..2...8..96..5....53..9...1..8...26....4...3......1..4......7..7...3..",
    "1...34.8....8..5....4.6..21.18......3..1.2..6......81.52..7.9....6..9....9.64...2",
    "...92......68.3...19..7...623..4.1....1...7....8.3..297...8..91...5.72......64...",
    ".6.5.4.3.1...9...8.........9...5...6.4.6.2.7.7...4...5.........4...8...1.5.2.3.4.",
    "7.....4...2..7..8...3..8.799..5..3...6..2..9...1.97..6...3..9...3..4..6...9..1.35",
];

// Source : Sudoku_hardest.txt ligne 11 (regime Expert 17-24 indices).
// Worst-case classique : backtracking exact timeout, recuit simule n'a aucune
// garantie de converger dans la fenetre de 5s/puzzle.
const EXPERT_PUZZLES: [&str; 1] = [
    "....7..2.8.......6.1.2.5...9.54....8.........3....85.1...3.2.8.4.......9.7..6....",
];

println!("Puzzles charges :");
println!("  Easy:   {} puzzles", EASY_PUZZLES.len());
println!("  Medium: {} puzzles", MEDIUM_PUZZLES.len());
println!("  Hard:   {} puzzles", HARD_PUZZLES.len());
println!("  Expert: {} puzzles", EXPERT_PUZZLES.len());

// Verification : compter les cases vides par niveau
for (name, puzzles) in [
    ("Easy", EASY_PUZZLES.as_slice()),
    ("Medium", MEDIUM_PUZZLES.as_slice()),
    ("Hard", HARD_PUZZLES.as_slice()),
    ("Expert", EXPERT_PUZZLES.as_slice()),
] {
    let empties: Vec<usize> = puzzles
        .iter()
        .map(|p| SudokuGrid::from_string(p).unwrap().count_empty())
        .collect();
    let min_e = *empties.iter().min().unwrap();
    let max_e = *empties.iter().max().unwrap();
    let moy: f64 = empties.iter().sum::<usize>() as f64 / empties.len() as f64;
    println!("  {}: {}-{} cases vides (moy: {:.1})", name, min_e, max_e, moy);
}

Puzzles charges :


  Easy:   10 puzzles


  Medium: 10 puzzles


  Hard:   10 puzzles


  Expert: 1 puzzles


  Easy: 45-57 cases vides (moy: 51.4)


  Medium: 64-64 cases vides (moy: 64.0)


  Hard: 53-59 cases vides (moy: 56.2)


  Expert: 59-59 cases vides (moy: 59.0)


()

### Interpretation : Jeux de test et profil des puzzles

**Repartition des jeux de test** :

| Difficulte | Nombre | Cases vides (min-max) | Moyenne |
|------------|--------|-----------------------|---------|
| Easy | 10 | 45-57 | 51.4 |
| Medium | 10 | 64 | 64.0 |
| Hard | 10 | 53-59 | 56.2 |

**Observation contre-intuitive** : Les puzzles Medium ont en moyenne **plus de cases vides** (64) que les puzzles Hard (56.2). Cela s'explique par la provenance des données :

- **Easy** : Grilles classiques avec beaucoup d'indices, la propagation suffit généralement
- **Medium** : Puzzles "top95" avec peu d'indices (presque toutes les cases vides), ce qui crée un espace de recherche immense
- **Hard** : Puzzles "hardest" connus pour leur difficulté logique, mais qui ont parfois plus d'indices que les Medium

> **Lecon** : La difficulté algorithmique ne depend pas seulement du nombre de cases vides, mais surtout de la **structure des contraintes**. Un puzzle avec 53 cases vides bien placees peut etre plus resistant qu'un puzzle avec 64 cases vides mal reparties.

**Protocole du benchmark** (aligné sur Sudoku-18b) :

- **5 grilles par difficulté** parmi les 10 disponibles — la variabilité *inter-grilles* devient mesurable ; **Expert (1 grille) est une étude de cas**, pas une distribution ;
- les runs résolus en moins de 250 ms sont **répétés 3 fois sur la même grille** : le spread des répétitions isole le **bruit système**, distinct de la variabilité inter-grilles ;
- timeout **coopératif** de 30 s par essai (`SolverTimeout`) — aucun thread orphelin ;
- un timeout est une **observation censurée** : comptée au dénominateur, exclue des médianes/IQR.

### Exécution du benchmark

La fonction `run_benchmark` teste chaque solveur sur chaque ensemble de puzzles et collecte les temps de résolution et le taux de succès. Chaque puzzle est résolu sur une **copie** de la grille pour éviter toute interference entre solveurs.

In [11]:
// Dictionnaires de solveurs et de puzzles pour le benchmark
// Trait commun : le dictionnaire Python devient Vec<(&str, Box<dyn SudokuSolver>)>.
trait SudokuSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError>;
    fn set_deadline(&mut self, deadline: Option<Instant>);
}

impl SudokuSolver for BacktrackingSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve(grid) // methode inharente (prioritaire sur la methode de trait)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

impl SudokuSolver for MRVBacktrackingSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve(grid)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

impl SudokuSolver for NorvigSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve(grid)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

impl SudokuSolver for DlxSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve(grid)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

let mut solvers: Vec<(&str, Box<dyn SudokuSolver>)> = vec![
    ("Backtracking", Box::new(BacktrackingSolver::new())),
    ("BT + MRV", Box::new(MRVBacktrackingSolver::new())),
    ("Norvig", Box::new(NorvigSolver::new())),
    ("DLX (remplace CP-SAT)", Box::new(DlxSolver::new())),
];

let puzzle_sets: Vec<(&str, Vec<&str>)> = vec![
    ("Easy", EASY_PUZZLES.to_vec()),
    ("Medium", MEDIUM_PUZZLES.to_vec()),
    ("Hard", HARD_PUZZLES.to_vec()),
    ("Expert", EXPERT_PUZZLES.to_vec()),
];

println!(
    "Solveurs configures : [{}]",
    solvers.iter().map(|(n, _)| *n).collect::<Vec<_>>().join(", ")
);
println!(
    "Puzzles disponibles : [{}]",
    puzzle_sets.iter().map(|(n, _)| *n).collect::<Vec<_>>().join(", ")
);

Solveurs configures : [Backtracking, BT + MRV, Norvig, DLX (remplace CP-SAT)]


Puzzles disponibles : [Easy, Medium, Hard, Expert]


### Configuration du benchmark

Les 4 solveurs et 3 niveaux de difficulté sont organises en vecteurs de tuples pour permettre une itération systématique. La fonction `run_benchmark` définie ci-dessous encapsule la logique de test avec gestion des timeouts.

In [12]:
use std::time::Duration;

// runs resolus < 250 ms : repetes pour isoler le bruit systeme
const FAST_REPEAT_MS: f64 = 250.0;
// repetitions additionnelles de la MEME grille
const N_REPEATS: usize = 3;

#[derive(Clone, Default)]
struct BenchStats {
    n_grids: usize,
    n_timeouts: usize,
    n_unsolved: usize,
    n_completed_runs: usize,
    solved: usize,
    count: usize,
    median_ms: f64,
    p25_ms: f64,
    p75_ms: f64,
    time_avg_ms: f64,
    time_total_ms: f64,
    times_ms: Vec<f64>,
}

/// Percentile par interpolation lineaire (comme numpy.percentile par defaut).
fn percentile(sorted: &[f64], p: f64) -> f64 {
    if sorted.is_empty() {
        return f64::NAN;
    }
    let n = sorted.len();
    if n == 1 {
        return sorted[0];
    }
    let rank = p * (n - 1) as f64;
    let lo = rank.floor() as usize;
    let hi = (lo + 1).min(n - 1);
    let frac = rank - lo as f64;
    sorted[lo] * (1.0 - frac) + sorted[hi] * frac
}

/// Benchmark statistiquement interpretable (protocole aligne sur Sudoku-18b).
///
/// - Plusieurs grilles par difficulte : la variabilite INTER-GRILLES est mesurable.
///   Expert ne contient qu'une grille : etude de cas, pas une distribution.
/// - Les runs resolus rapides (< FAST_REPEAT_MS) sont repetes N_REPEATS fois sur
///   la meme grille : le spread des repetitions isole le BRUIT SYSTEME.
/// - Timeout COOPERATIF : chaque solveur consulte sa deadline et renvoie
///   Err(SolverError::Timeout) -- aucun thread orphelin ne survit au budget.
/// - Censure : un timeout est compte au denominateur (taux de resolution) mais
///   EXCLU des statistiques de temps (mediane, IQR).
fn run_benchmark<'a>(
    solvers: &mut [(&str, Box<dyn SudokuSolver>)],
    puzzle_sets: &[(&'a str, Vec<&'a str>)],
    grids_per_diff: usize,
    timeout_s: f64,
) -> Vec<(String, Vec<(&'a str, BenchStats)>)> {
    let mut results: Vec<(String, Vec<(&'a str, BenchStats)>)> = Vec::new();
    for (solver_name, solver) in solvers.iter_mut() {
        let mut per_diff: Vec<(&'a str, BenchStats)> = Vec::new();
        for (diff_name, puzzles) in puzzle_sets {
            let n_grids = grids_per_diff.min(puzzles.len());
            let mut completed_ms: Vec<f64> = Vec::new();
            let mut n_timeouts = 0usize;
            let mut n_unsolved = 0usize;
            for puzzle_str in puzzles.iter().take(n_grids) {
                let grid = SudokuGrid::from_string(puzzle_str).unwrap();
                let mut run_times: Vec<f64> = Vec::new();
                let mut timed_out = false;
                solver.set_deadline(Some(Instant::now() + Duration::from_secs_f64(timeout_s)));
                let mut g = grid.clone();
                let start = Instant::now();
                match solver.solve(&mut g) {
                    Ok(true) => {
                        let elapsed = start.elapsed().as_secs_f64() * 1000.0;
                        run_times.push(elapsed);
                        if elapsed < FAST_REPEAT_MS {
                            for _ in 0..N_REPEATS {
                                solver.set_deadline(Some(
                                    Instant::now() + Duration::from_secs_f64(timeout_s),
                                ));
                                let mut g2 = grid.clone();
                                let s2 = Instant::now();
                                if let Ok(true) = solver.solve(&mut g2) {
                                    run_times.push(s2.elapsed().as_secs_f64() * 1000.0);
                                }
                            }
                        }
                    }
                    Ok(false) => n_unsolved += 1,
                    Err(SolverError::Timeout) => timed_out = true,
                }
                solver.set_deadline(None); // finally : budget rendu
                if timed_out {
                    n_timeouts += 1;
                } else {
                    completed_ms.extend(run_times);
                }
            }
            let mut arr = completed_ms.clone();
            arr.sort_by(|a, b| a.partial_cmp(b).unwrap());
            let n_runs = completed_ms.len();
            let total: f64 = completed_ms.iter().sum();
            per_diff.push((
                *diff_name,
                BenchStats {
                    n_grids,
                    n_timeouts,
                    n_unsolved,
                    n_completed_runs: n_runs,
                    solved: n_grids - n_timeouts - n_unsolved,
                    count: n_grids,
                    median_ms: percentile(&arr, 0.5),
                    p25_ms: percentile(&arr, 0.25),
                    p75_ms: percentile(&arr, 0.75),
                    time_avg_ms: if n_runs == 0 { f64::NAN } else { total / n_runs as f64 },
                    time_total_ms: total,
                    times_ms: completed_ms,
                },
            ));
        }
        results.push((solver_name.to_string(), per_diff));
    }
    results
}

println!("Fonction definie : run_benchmark (multi-grilles, repetitions, censure, timeout cooperatif)");

// Smoke : 2 solveurs, 1 grille Easy (repetitions) + 1 Expert (censure eventuelle)
let mut smoke_solvers: Vec<(&str, Box<dyn SudokuSolver>)> = vec![
    ("Backtracking", Box::new(BacktrackingSolver::new())),
    ("Norvig", Box::new(NorvigSolver::new())),
];
let smoke_sets: Vec<(&str, Vec<&str>)> = vec![
    ("Easy", vec![EASY_PUZZLES[0]]),
    ("Expert", vec![EXPERT_PUZZLES[0]]),
];
let smoke = run_benchmark(&mut smoke_solvers, &smoke_sets, 1, 5.0);
for (name, per) in &smoke {
    for (d, st) in per {
        println!(
            "SMOKE {} {} : resolus {}/{}, TO {}, mediane {:.2} ms, {} runs completes",
            name, d, st.solved, st.count, st.n_timeouts, st.median_ms, st.n_completed_runs
        );
    }
}

Fonction definie : run_benchmark (multi-grilles, repetitions, censure, timeout cooperatif)


SMOKE Backtracking Easy : resolus 1/1, TO 0, mediane 0.02 ms, 4 runs completes


SMOKE Backtracking Expert : resolus 1/1, TO 0, mediane 17.16 ms, 4 runs completes


SMOKE Norvig Easy : resolus 1/1, TO 0, mediane 0.46 ms, 4 runs completes


SMOKE Norvig Expert : resolus 1/1, TO 0, mediane 0.69 ms, 4 runs completes


()

### Lancement du benchmark

Nous executons le benchmark statistique sur les solveurs du framework : 5 grilles par difficulte, repetitions des runs rapides, et un timeout **cooperatif** de 30 secondes par essai -- chaque solveur consulte une echeance et leve `SolverTimeout`, le calcul est reellement interrompu (aucun thread orphelin ne survit au budget).

In [13]:
// Lancer le benchmark statistique : 5 grilles par difficulte (Expert : 1 grille,
// etude de cas), timeout cooperatif 30 s PAR ESSAI, repetitions des runs rapides.
println!("Benchmark en cours (5 grilles/difficulte, timeout cooperatif 30 s/essai, repetitions des runs < 250 ms)...");
let results = run_benchmark(&mut solvers, &puzzle_sets, 5, 30.0);
println!("Termine.");

Benchmark en cours (5 grilles/difficulte, timeout cooperatif 30 s/essai, repetitions des runs < 250 ms)...


Termine.


### Tableau recapitulatif des résultats

Affichons les résultats sous forme de tableau synthetique, puis sous forme graphique.

> **Note** : Le benchmark exécute 5 grilles par difficulté (Expert : 1 grille, étude de cas) avec un timeout **cooperatif** de 30 s par essai — chaque solveur consulte une échéance et lève `SolverTimeout`, aucun thread orphelin ne survit au budget (en Python, l'ancien `ThreadPoolExecutor` attendait le thread non annulable : un timeout de 30 s pouvait durer 69 s réels ; le portage Rust lève `SolverError::Timeout` directement). Les timeouts sont des observations **censurées** : comptées au dénominateur du taux de résolution, exclues des statistiques de temps. exécution rapide tout en montrant les différences de performance entre solveurs.

In [14]:
fn variability_label(res: &BenchStats) -> String {
    // Qualificatif DERIVE des mesures (jamais hand-written), seuils explicites.
    if res.n_grids < 2 {
        return "etude de cas (1 grille) : repetitions = bruit systeme".to_string();
    }
    if res.n_completed_runs < 3 || !res.median_ms.is_finite() || res.median_ms <= 0.0 {
        return "censure totale : pas de distribution".to_string();
    }
    let ratio = (res.p75_ms - res.p25_ms) / res.median_ms;
    if ratio < 0.25 {
        format!("tres faible (IQR/med = {:.2})", ratio)
    } else if ratio < 0.5 {
        format!("faible (IQR/med = {:.2})", ratio)
    } else if ratio < 1.0 {
        format!("moderee (IQR/med = {:.2})", ratio)
    } else {
        format!("forte (IQR/med = {:.2})", ratio)
    }
}

fn stat_for<'a>(
    results: &'a [(String, Vec<(&'a str, BenchStats)>)],
    solver: &str,
    diff: &str,
) -> Option<&'a BenchStats> {
    results
        .iter()
        .find(|(n, _)| n == solver)?
        .1
        .iter()
        .find(|(d, _)| *d == diff)
        .map(|(_, s)| s)
}

fn print_benchmark_table(results: &[(String, Vec<(&str, BenchStats)>)]) {
    // Mediane [p25-p75], taux AVEC denominateur, timeouts censurees, effectifs.
    let header = format!(
        "{:<18} {:<10} {:>24} {:>7} {:>4} {:>5}  Variabilite",
        "Solveur", "Difficulte", "Mediane [p25-p75] ms", "Resolu", "TO", "Runs"
    );
    println!("{}", header);
    println!("{}", "-".repeat(header.len()));
    for (solver_name, per_diff) in results {
        for (diff, r) in per_diff {
            let cell_t = if r.n_completed_runs > 0 && r.median_ms.is_finite() {
                format!("{:>10.1} [{:.1}-{:.1}]", r.median_ms, r.p25_ms, r.p75_ms)
            } else {
                format!("{:>24}", "(censure)")
            };
            println!(
                "{:<18} {:<10} {:>24} {:>3}/{:<3} {:>4} {:>5}  {}",
                solver_name, diff, cell_t, r.solved, r.count, r.n_timeouts,
                r.n_completed_runs, variability_label(r)
            );
        }
    }
}

println!("=== Resultats du benchmark (mediane [p25-p75] ; TO = timeouts censurees) ===\n");
print_benchmark_table(&results);

// Ratios derives des medianes fraiches (jamais d'un run unique)
let difficulties = ["Easy", "Medium", "Hard", "Expert"];
println!("\n--- Ratios de vitesse (medianes, entre solveurs ayant complete >= 1 run) ---");
for diff in difficulties {
    let mut meds: Vec<(&str, f64, usize)> = Vec::new();
    for (s, _) in &results {
        if let Some(r) = stat_for(&results, s, diff) {
            if r.n_completed_runs > 0 && r.median_ms.is_finite() {
                meds.push((s.as_str(), r.median_ms, r.n_completed_runs));
            }
        }
    }
    if meds.len() >= 2 {
        let (fast, slow) = {
            let mut it = meds.iter();
            let mut f = it.next().unwrap();
            let mut s = f;
            for m in it {
                if m.1 < f.1 { f = m; }
                if m.1 >= s.1 { s = m; }
            }
            (f, s)
        };
        println!(
            "  {:<8}: {} ({} runs) vs {} ({} runs) -> x{:.1}",
            diff, fast.0, fast.2, slow.0, slow.2, slow.1 / fast.1
        );
    }
}

println!("\n--- Resolution globale (denominateurs conserves) ---");
for (solver_name, per_diff) in &results {
    let tot: usize = per_diff.iter().map(|(_, s)| s.count).sum();
    let sol: usize = per_diff.iter().map(|(_, s)| s.solved).sum();
    let to: usize = per_diff.iter().map(|(_, s)| s.n_timeouts).sum();
    println!("  {:<18}: {}/{} resolus, {} timeouts censurees", solver_name, sol, tot, to);
}

=== Resultats du benchmark (mediane [p25-p75] ; TO = timeouts censurees) ===


Solveur            Difficulte     Mediane [p25-p75] ms  Resolu   TO  Runs  Variabilite


--------------------------------------------------------------------------------------


Backtracking       Easy                  0.1 [0.0-0.2]   5/5      0    20  forte (IQR/med = 2.90)


Backtracking       Medium            49.6 [12.0-256.6]   5/5      0    11  forte (IQR/med = 4.94)


Backtracking       Hard               32.1 [20.9-44.4]   5/5      0    20  moderee (IQR/med = 0.73)


... (20 outputs left out)

  BT + MRV          : 16/16 resolus, 0 timeouts censurees


  Norvig            : 16/16 resolus, 0 timeouts censurees


  DLX (remplace CP-SAT): 16/16 resolus, 0 timeouts censurees


()

## Exercice : Analyser la robustesse des solveurs face aux puzzles partiels

**Objectif**
Testez comment chaque solveur se comporte quand on augmente progressivement
le nombre de cases vides dans un puzzle (de 30 a 55 cases vides).

**Indice**
Partez d'un puzzle résolu et supprimez aléatoirement un nombre croissant de valeurs.
Mesurez le temps de résolution pour chaque niveau de difficulté artificielle.


In [15]:
// EXERCICE : Analyser la robustesse des solveurs face aux puzzles partiels
fn measure_robustness(
    _solvers: &mut [(&str, Box<dyn SudokuSolver>)],
    _base_puzzle: &str,
    _removal_counts: &[usize],
) -> std::collections::BTreeMap<String, Vec<Option<f64>>> {
    // TODO: Pour chaque nombre de suppressions, creez un puzzle partiel
    // et testez chaque solveur. Retournez les temps de resolution.
    let result: std::collections::BTreeMap<String, Vec<Option<f64>>> = std::collections::BTreeMap::new(); // TODO etudiant
    result
}

println!("Exercice a completer : measure_robustness");

// Sonde figure inline (plotters feature evcxr)
use plotters::prelude::*;
use plotters::evcxr::evcxr_figure;

let probe = evcxr_figure((640, 200), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("probe evcxr", ("sans-serif", 16))
        .build_cartesian_2d(0i32..10i32, 0i32..10i32)?;
    chart.draw_series(std::iter::once(Circle::new((5, 5), 5, &RED)))?;
    Ok(())
})
.style("width:60%");
probe

Exercice a completer : measure_robustness


probe evcxr

### Interpretation : Résultats du benchmark

Lecture du tableau (médiane [p25-p75], taux **avec dénominateur**, timeouts censurées) :

1. **Backtracking simple** : en Rust compilé, il complète ici tout le benchmark (16/16, zéro timeout) — mais avec les médianes les plus hautes et la plus forte dispersion. La censure des timeouts reste pertinente pour le recuit simulé de la section extension.

2. **BT + MRV** : l'heuristique MRV réduit fortement l'arbre. Comparer ses médianes à celles du backtracking simple reste toutefois biaisé : les grilles que le simple ne termine pas sont précisément les plus dures (biais de censure) — d'où le taux de résolution affiché à côté du temps.

3. **Norvig** et **DLX** : les deux moteurs complètent l'ensemble ; leurs IQR quantifient la variabilité inter-grilles, et les répétitions des runs rapides donnent le bruit système.

**Règle de lecture** : une médiane ne se compare entre solveurs que si les taux de résolution sont égaux — sinon la comparaison favorise mécaniquement le solveur qui n'a tenté que les grilles faciles. C'est le protocole de Sudoku-18b (bootstrap, Mann-Whitney, comparaisons multiples) appliqué au benchmark des twins.


### Visualisation : mediane par difficulte

Le graphique en barres groupees montre la **mediane** de resolution pour chaque solveur sur chaque niveau de difficulte, avec les barres d'erreur p25-p75. L'echelle logarithmique (substitut du symlog matplotlib, absent de plotters) permet de comparer des ordres de grandeur très différents ; une barre nulle annotee "0/n TO" signale une difficulte entierement censuree (timeouts), information en soi.

In [16]:
use plotters::coord::Shift;
use plotters::element::{DynElement, PathElement, Rectangle, Text};
use plotters::style::ShapeStyle;

fn plot_benchmark_bars(
    results: &[(String, Vec<(&str, BenchStats)>)],
) -> plotters::evcxr::SVGWrapper {
    // Barres groupees : MEDIANE par difficulte, barres d'erreur p25-p75.
    // Une difficulte entierement censuree (0/n) est montree en barre nulle annotee.
    // NOTE : echelle LOGARITHMIQUE (plotters n'a pas de symlog natif) ; les
    // valeurs <= 0 sont clampes au plancher d'axe 0.05 ms.
    let difficulties = ["Easy", "Medium", "Hard", "Expert"];
    let colors = [
        RGBColor(0x4C, 0x72, 0xB0),
        RGBColor(0x55, 0xA8, 0x68),
        RGBColor(0xC4, 0x4E, 0x52),
        RGBColor(0x81, 0x72, 0xB2),
        RGBColor(0xFF, 0x8C, 0x00),
    ];

    evcxr_figure((1100, 600), move |root| {
        root.fill(&WHITE)?;
        let font = ("sans-serif", 14).into_font();
        let small = ("sans-serif", 11).into_font();
        let tiny = ("sans-serif", 10).into_font();

        // Marges et zone de tracé
        let (left, right, top, bottom) = (80i32, 20, 70, 70);
        let plot_w = 1100 - left - right; // 1000
        let plot_h = 600 - top - bottom; // 460
        let group_w = plot_w / 4;
        let n_solvers = results.len() as i32;
        let bar_w = ((0.8 * group_w as f64) / n_solvers as f64) as i32;

        // Echelle log : plancher 0.05 ms, plafond = decenie au-dessus du max des p75
        let y_floor = 0.05f64;
        let mut max_v = 0.1f64;
        for (_, per) in results {
            for (_, r) in per {
                if r.n_completed_runs > 0 && r.p75_ms > max_v {
                    max_v = r.p75_ms;
                }
            }
        }
        let y_top = 10f64.powf(max_v.log10().ceil());
        let lf = y_floor.log10();
        let lt = y_top.log10();
        let py = |v: f64| -> i32 {
            let c = v.max(y_floor).min(y_top);
            top + ((1.0 - (c.log10() - lf) / (lt - lf)) * plot_h as f64) as i32
        };

        // Grille + graduations Y (decenies)
        let mut tick = y_floor;
        while tick <= y_top * 1.0001 {
            let y = py(tick);
            root.draw(&PathElement::new(
                [(left, y), (left + plot_w, y)],
                ShapeStyle::from(&RGBColor(0xE0, 0xE0, 0xE0)),
            ))?;
            let lab = if tick >= 1.0 { format!("{}", tick as i64) } else { format!("{}", tick) };
            root.draw(&Text::new(lab, (left - 40, y - 6), small.clone()))?;
            tick *= 10.0;
        }
        root.draw(&Text::new("Mediane (ms) - echelle log".to_string(), (0, 6), small.clone()))?;

        // Titre
        root.draw(&Text::new(
            "Benchmark : mediane par solveur (barres d erreur = p25-p75 ; Expert = etude de cas, 1 grille)".to_string(),
            (280, 20), font.clone(),
        ))?;

        // Barres
        for (gi, diff) in difficulties.iter().enumerate() {
            let group_cx = left + group_w * gi as i32 + group_w / 2;
            // Nom de difficulte sous le groupe
            root.draw(&Text::new(diff.to_string(), (group_cx - 18, top + plot_h + 18), font.clone()))?;
            for (si, (solver_name, per)) in results.iter().enumerate() {
                let r = per.iter().find(|(d, _)| d == diff).map(|(_, s)| s).unwrap();
                let offset = (si as i32 - n_solvers / 2) * bar_w + if n_solvers % 2 == 0 { bar_w / 2 } else { 0 };
                let cx = group_cx + offset;
                let color = &colors[si % colors.len()];
                let (med, label) = if r.n_completed_runs > 0 && r.median_ms.is_finite() {
                    (r.median_ms, format!("{:.1}", r.median_ms))
                } else {
                    (0.0, format!("{}/{} TO", r.count - r.solved - r.n_timeouts, r.count))
                };
                let (y_base, y_med) = (py(y_floor), py(med));
                if med > 0.0 {
                    root.draw(&Rectangle::new(
                        [(cx - bar_w / 2 + 1, y_med), (cx + bar_w / 2 - 1, y_base)],
                        ShapeStyle::from(color).filled(),
                    ))?;
                    // Barre d'erreur p25-p75 + coiffes
                    if r.n_completed_runs > 0 {
                        let (y_lo, y_hi) = (py(r.p75_ms), py(r.p25_ms));
                        let style = ShapeStyle::from(&BLACK);
                        root.draw(&PathElement::new([(cx, y_lo), (cx, y_hi)], style.clone()))?;
                        root.draw(&PathElement::new([(cx - 4, y_lo), (cx + 4, y_lo)], style.clone()))?;
                        root.draw(&PathElement::new([(cx - 4, y_hi), (cx + 4, y_hi)], style.clone()))?;
                    }
                }
                // Valeur au-dessus de la barre (ou "0/n TO")
                let y_lab = if med > 0.0 { py(med) - 18 } else { py(y_floor) - 18 };
                root.draw(&Text::new(label, (cx - 16, y_lab), tiny.clone()))?;
            }
        }

        // Legende
        for (si, (solver_name, _)) in results.iter().enumerate() {
            let y = top + 8 + si as i32 * 22;
            root.draw(&Rectangle::new(
                [(left + 4, y), (left + 20, y + 12)],
                ShapeStyle::from(&colors[si % colors.len()]).filled(),
            ))?;
            root.draw(&Text::new(solver_name.clone(), (left + 26, y), small.clone()))?;
        }
        Ok(())
    })
    .style("width:100%")
}

plot_benchmark_bars(&results)

0.05
 
 
 
0.5
 
 
 
5
 
 
 
50
 
 
 
500
 
 
Mediane (ms) - echelle log
 
 
Benchmark : mediane par solveur (barres d erreur = p25-p75 ; Expert = etude de cas, 1 grille)
 
 
Easy
 
 
 
 
 
 
0.1
 
 
 
 
 
 
0.2
 
 
 
 
 
 
0.5
 
 
 
 
 
 
0.6
 
 
Medium
 
 
 
 
 
 
49.6
 
 
 
 
 
 
32.2
 
 
 
 
 
 
2.0
 
 
 
 
 
 
1.3
 
 
Hard
 
 
 
 
 
 
32.1
 
 
 
 
 
 
2.9
 
 
 
 
 
 
0.7
 
 
 
 
 
 
0.8
 
 
Expert
 
 
 
 
 
 
13.3
 
 
 
 
 
 
2.9
 
 
 
 
 
 
0.8
 
 
 
 
 
 
0.8
 
 
 
Backtracking
 
 
 
BT + MRV
 
 
 
Norvig
 
 
 
DLX (remplace CP-SAT)

### Interpretation : Ecarts de performance entre solveurs

Les ratios affichés sous le tableau sont **dérivés des médianes fraîches** (avec leurs effectifs), plus d'un ratio ponctuel mono-grille :

- un ratio n'est interprétable que si les deux solveurs ont complété les **mêmes grilles** (mêmes taux) ;
- sur une difficulté où un solveur est 0/n (tout censuré), le ratio est absent **par construction** — c'est une information (explosion), pas un manque ;
- l'échelle log des barres rend lisibles les ordres de grandeur **et** les barres nulles (0/n timeouts).

**Point remarquable** : les puzzles Medium (top95, ~64 cases vides) restent les plus discriminants pour les solveurs naïfs ; les puzzles Hard (hardest) ont parfois moins de cases vides mais des contraintes plus serrées qui guident mieux la propagation.


### Visualisation : distribution des temps par solveur

Les boites a moustaches (boxplots) montrent la **variabilite** des temps de résolution. Un solveur avec une boite etroite est plus **previsible** dans ses performances. (Les moustaches s'étendent aux extrêmes min-max : le portage plotters ne clippe pas à 1.5xIQR comme matplotlib.)

In [17]:
fn plot_boxplots(results: &[(String, Vec<(&str, BenchStats)>)]) -> plotters::evcxr::SVGWrapper {
    // Boxplots par difficulté -- UNIQUEMENT si >= 3 runs complétés ET >= 2 grilles.
    // Expert (1 grille) : nuage de points (étude de cas), répétitions = bruit système.
    // NOTE : moustaches min->max (la boxplot matplotlib clippe à 1.5*IQR et trace
    // les outliers ; ici les points extrêmes restent dans la moustache, documenté).
    let difficulties = ["Easy", "Medium", "Hard", "Expert"];
    let colors = [
        RGBColor(0x4C, 0x72, 0xB0),
        RGBColor(0x55, 0xA8, 0x68),
        RGBColor(0xC4, 0x4E, 0x52),
        RGBColor(0x81, 0x72, 0xB2),
        RGBColor(0xFF, 0x8C, 0x00),
    ];

    evcxr_figure((1100, 500), move |root| {
        root.fill(&WHITE)?;
        let small = ("sans-serif", 10).into_font();

        let margin_l: i32 = 60;
        let panel_gap: i32 = 24;
        let panel_w: i32 = (1100 - margin_l - 30 - panel_gap * 3) / 4;
        let top: i32 = 60;
        let plot_h: i32 = 340;

        root.draw(&Text::new(
            "Inter-grilles par boxplot ; Expert = répétitions (bruit système), pas une distribution".to_string(),
            (220, 8), ("sans-serif", 13).into_font(),
        ))?;

        for (gi, diff) in difficulties.iter().enumerate() {
            let px0: i32 = margin_l + gi as i32 * (panel_w + panel_gap);
            let py0: i32 = top;
            let px1: i32 = px0 + panel_w;
            let py1: i32 = py0 + plot_h;

            let mut vmax = 1e-9f64;
            for (_, per) in results {
                if let Some((_, r)) = per.iter().find(|(d, _)| d == diff) {
                    if r.n_completed_runs > 0 {
                        for t in &r.times_ms {
                            if *t > vmax {
                                vmax = *t;
                            }
                        }
                    }
                }
            }
            vmax *= 1.05;
            let py_of = |v: f64| -> i32 { py1 - ((v / vmax) * plot_h as f64) as i32 };

            let frame = ShapeStyle::from(&RGBColor(0xE0, 0xE0, 0xE0));
            for k in 0..=4 {
                let v = vmax * k as f64 / 4.0;
                let y = py_of(v);
                root.draw(&PathElement::new([(px0, y), (px1, y)], frame.clone()))?;
                root.draw(&Text::new(format!("{:.0}", v), (px0 - 34, y - 5), small.clone()))?;
            }

            let title = if *diff == "Expert" {
                "Expert : étude de cas (1 grille)".to_string()
            } else {
                format!("{} (boxplot si n >= 3 runs, >= 2 grilles)", diff)
            };
            root.draw(&Text::new(title, (px0 + 6, py0 - 22), small.clone()))?;

            let n = results.len() as i32;
            let slot = panel_w / n;
            for (si, (solver_name, per)) in results.iter().enumerate() {
                let r = per.iter().find(|(d, _)| d == diff).map(|(_, s)| s).unwrap();
                let color = &colors[si % colors.len()];
                let cx: i32 = px0 + slot * si as i32 + slot / 2;

                if *diff == "Expert" {
                    if r.n_completed_runs > 0 {
                        for t in &r.times_ms {
                            root.draw(&Circle::new((cx, py_of(*t)), 3, color.filled()))?;
                        }
                    }
                } else if r.n_grids >= 2 && r.n_completed_runs >= 3 {
                    let bw = slot * 2 / 5;
                    let mut sorted = r.times_ms.clone();
                    sorted.sort_by(|a, b| a.partial_cmp(b).unwrap());
                    let q = |p: f64| percentile(&sorted, p);
                    let (p25, med, p75) = (q(0.25), q(0.5), q(0.75));
                    let (mn, mx) = (sorted[0], sorted[sorted.len() - 1]);
                    let whisk = ShapeStyle::from(&RGBColor(0x60, 0x60, 0x60));
                    root.draw(&PathElement::new([(cx, py_of(mn)), (cx, py_of(p25))], whisk.clone()))?;
                    root.draw(&PathElement::new([(cx - 5, py_of(mn)), (cx + 5, py_of(mn))], whisk.clone()))?;
                    root.draw(&PathElement::new([(cx, py_of(p75)), (cx, py_of(mx))], whisk.clone()))?;
                    root.draw(&PathElement::new([(cx - 5, py_of(mx)), (cx + 5, py_of(mx))], whisk.clone()))?;
                    let mixed = RGBColor(
                        (color.0 / 4 * 3 + 255 / 4) as u8,
                        (color.1 / 4 * 3 + 255 / 4) as u8,
                        (color.2 / 4 * 3 + 255 / 4) as u8,
                    );
                    root.draw(&Rectangle::new(
                        [(cx - bw / 2, py_of(p75)), (cx + bw / 2, py_of(p25))],
                        ShapeStyle::from(&mixed).filled(),
                    ))?;
                    root.draw(&Rectangle::new(
                        [(cx - bw / 2, py_of(p75)), (cx + bw / 2, py_of(p25))],
                        ShapeStyle::from(color),
                    ))?;
                    root.draw(&PathElement::new(
                        [(cx - bw / 2, py_of(med)), (cx + bw / 2, py_of(med))],
                        ShapeStyle::from(&BLACK).stroke_width(2),
                    ))?;
                }

                let short: String = solver_name.chars().take(12).collect();
                root.draw(&Text::new(short, (cx - 24, py1 + 16), ("sans-serif", 9).into_font()))?;
            }
        }
        Ok(())
    })
    .style("width:100%")
}

plot_boxplots(&results)

Inter-grilles par boxplot ; Expert = répétitions (bruit système), pas une distribution
 
 
 
0
 
 
 
1
 
 
 
2
 
 
 
2
 
 
 
3
 
 
Easy (boxplot si n >= 3 runs, >= 2 grilles)
 
 
 
 
 
 
 
 
 
Backtracking
 
 
 
 
 
 
 
 
 
BT + MRV
 
 
 
 
 
 
 
 
 
Norvig
 
 
 
 
 
 
 
 
 
DLX (remplac
 
 
 
0
 
 
 
4885
 
 
 
9770
 
 
 
14655
 
 
 
19539
 
 
Medium (boxplot si n >= 3 runs, >= 2 grilles)
 
 
 
 
 
 
 
 
 
Backtracking
 
 
 
 
 
 
 
 
 
BT + MRV
 
 
 
 
 
 
 
 
 
Norvig
 
 
 
 
 
 
 
 
 
DLX (remplac
 
 
 
0
 
 
 
48
 
 
 
95
 
 
 
143
 
 
 
190
 
 
Hard (boxplot si n >= 3 runs, >= 2 grilles)
 
 
 
 
 
 
 
 
 
Backtracking
 
 
 
 
 
 
 
 
 
BT + MRV
 
 
 
 
 
 
 
 
 
Norvig
 
 
 
 
 
 
 
 
 
DLX (remplac
 
 
 
0
 
 
 
4
 
 
 
7
 
 
 
11
 
 
 
14
 
 
Expert : étude de cas (1 grille)
 
 
 
 
 
 
Backtracking
 
 
 
 
 
 
BT + MRV
 
 
 
 
 
 
Norvig
 
 
 
 
 
 
DLX (remplac

### Interpretation : Variabilite et previsibilite des solveurs

Les qualificatifs du tableau sont **calculés depuis les mesures** (IQR/médiane, seuils explicites), plus hand-written sur une observation unique :

- **variabilité inter-grilles** : l'IQR des runs complétés — boxplots uniquement quand n >= 3 runs sur >= 2 grilles ;
- **bruit système** : le spread des répétitions d'une même grille rapide ;
- **Expert (1 grille)** : étude de cas — les points répétés mesurent le bruit système, **pas** une distribution.

**Leçon pratique** : en production, la prévisibilité compte autant que la vitesse médiane — un solveur 0/n sur une difficulté (timeouts censurées) est un risque opérationnel même si sa médiane sur les grilles complétées est excellente. Sudoku-18b formalise ce protocole (bootstrap, Mann-Whitney, corrections multi-comparaisons).


### Visualisation : radar multi-critères

Le graphique radar compare les solveurs sur **5 dimensions** qualitatives. Les scores sont normalises de 1 (faible) a 5 (excellent) pour permettre une vue d'ensemble.

| Critere | Description |
|---------|-------------|
| **Vitesse** | Temps de résolution moyen |
| **Fiabilite** | Taux de résolution sur tous les niveaux |
| **Scalabilite** | Robustesse face a la difficulté croissante |
| **Simplicite** | Nombre de lignes de code, facilite de comprehension |
| **Generalite** | Capacité a s'adapter a d'autrès problèmes que le Sudoku |

## Exemple : Ajouter un Solveur et Etendre le Benchmark

Cet exemple montre comment etendre le benchmark en ajoutant un cinquieme solveur. Nous implementons ici une variante du solveur Norvig avec du **forward checking explicite**.

### Principe du forward checking

Après chaque assignation tentative dans le backtracking, on vérifie immédiatement que toutes les cases vides ont encore au moins un candidat valide. Si une case a un domaine vide, la branche est coupee sans appel récursif supplementaire.

### Implementation

Le solveur `NorvigForwardCheckingSolver` encapsule le solveur Norvig standard (composition — pas d'héritage en Rust) en ajoutant :
- Une méthode `forward_check` qui parcourt toutes les cases et detecte les domaines vides
- Un appel a `forward_check` dans `search` juste après `assign`, avant de recurser
- Un compteur `fc_cuts` pour mesurer combien de branches sont coupees par le forward checking


In [18]:
// --- Cellule 46 : Norvig + forward checking explicite (Option B) ---

struct NorvigForwardCheckingSolver {
    deadline: Option<Instant>,
    call_count: usize,
    fc_cuts: usize,
    units: Vec<[usize; 9]>,
    units_of: [[usize; 3]; 81],
    peers: Vec<[usize; 20]>,
}

impl NorvigForwardCheckingSolver {
    // Principe : apres chaque assignation tentative dans le backtracking,
    // on verifie que toutes les cases vides ont encore au moins un candidat.
    // Si une case a un domaine vide, on echoue immediatement sans recurser.
    fn new() -> Self {
        let mut unitlist: Vec<[usize; 9]> = Vec::with_capacity(27);
        for r in 0..9 {
            let mut u = [0usize; 9];
            for c in 0..9 { u[c] = r * 9 + c; }
            unitlist.push(u);
        }
        for c in 0..9 {
            let mut u = [0usize; 9];
            for r in 0..9 { u[r] = r * 9 + c; }
            unitlist.push(u);
        }
        for br in 0..3 {
            for bc in 0..3 {
                let mut u = [0usize; 9];
                let mut k = 0;
                for r in 0..3 {
                    for c in 0..3 { u[k] = (br * 3 + r) * 9 + bc * 3 + c; k += 1; }
                }
                unitlist.push(u);
            }
        }
        let mut units_of = [[0usize; 3]; 81];
        for (ui, u) in unitlist.iter().enumerate() {
            for &s in u {
                let mut j = 0;
                while units_of[s][j] != 0 { j += 1; }
                units_of[s][j] = ui;
            }
        }
        let mut peers = Vec::with_capacity(81);
        for s in 0..81 {
            let mut set = [usize::MAX; 27];
            for &ui in &units_of[s] {
                for &s2 in &unitlist[ui] {
                    if s2 != s {
                        let mut j = 0;
                        while set[j] != usize::MAX && set[j] != s2 { j += 1; }
                        set[j] = s2;
                    }
                }
            }
            let mut arr = [0usize; 20];
            let mut n = 0;
            for &v in &set {
                if v != usize::MAX { arr[n] = v; n += 1; }
            }
            peers.push(arr);
        }
        NorvigForwardCheckingSolver { deadline: None, call_count: 0, fc_cuts: 0, units: unitlist, units_of, peers }
    }

    fn assign(&self, values: &mut [u16; 81], s: usize, d: u8) -> Result<(), ()> {
        // Assigne d a s en eliminant toutes les autres valeurs.
        let others = values[s] & !cand_bit(d);
        let mut bit = 1u16;
        while bit <= 0b1_1111_1111 {
            if others & bit != 0 {
                let d2 = bit.trailing_zeros() as u8 + 1;
                self.eliminate(values, s, d2)?;
            }
            bit <<= 1;
        }
        Ok(())
    }

    fn eliminate(&self, values: &mut [u16; 81], s: usize, d: u8) -> Result<(), ()> {
        // Elimine d des candidats de s et propage.
        if values[s] & cand_bit(d) == 0 {
            return Ok(());
        }
        values[s] &= !cand_bit(d);
        if values[s] == 0 {
            return Err(());
        }
        if values[s].count_ones() == 1 {
            let d2 = values[s].trailing_zeros() as u8 + 1;
            for &s2 in &self.peers[s] {
                self.eliminate(values, s2, d2)?;
            }
        }
        for &ui in &self.units_of[s] {
            let mut dplaces: Vec<usize> = Vec::new();
            for &s2 in &self.units[ui] {
                if values[s2] & cand_bit(d) != 0 { dplaces.push(s2); }
            }
            if dplaces.is_empty() { return Err(()); }
            if dplaces.len() == 1 {
                self.assign(values, dplaces[0], d)?;
            }
        }
        Ok(())
    }

    fn forward_check(&self, values: &[u16; 81]) -> bool {
        // Forward checking : toutes les cases doivent avoir au moins un candidat.
        values.iter().all(|&c| c != 0)
    }

    fn search(&mut self, values: [u16; 81]) -> Result<Option<[u16; 81]>, SolverError> {
        // Recherche DFS avec MRV + forward checking avant recursion.
        self.call_count += 1;
        if let Some(dl) = self.deadline {
            if (self.call_count & 255) == 0 && Instant::now() > dl {
                return Err(SolverError::Timeout);
            }
        }
        if values.iter().all(|&c| c.count_ones() == 1) {
            return Ok(Some(values));
        }
        let mut best = usize::MAX;
        let mut best_n = 10usize;
        for s in 0..81 {
            let n = values[s].count_ones() as usize;
            if n > 1 && n < best_n {
                best_n = n;
                best = s;
            }
        }
        let s = best;
        let mut d = 1u8;
        while d <= 9 {
            if values[s] & cand_bit(d) != 0 {
                let mut nv = values;
                let assigned = self.assign(&mut nv, s, d);
                if assigned.is_err() || !self.forward_check(&nv) {
                    self.fc_cuts += 1;
                    d += 1;
                    continue;
                }
                if let Some(sol) = self.search(nv)? {
                    return Ok(Some(sol));
                }
            }
            d += 1;
        }
        Ok(None)
    }
}

impl NorvigForwardCheckingSolver {
    fn solve_grid(&mut self, grid: &mut SudokuGrid) -> Result<bool, ()> {
        self.call_count = 0;
        self.fc_cuts = 0;
        let mut values = [FULL_CANDIDATES; 81];
        for i in 0..81 {
            let d = grid.cells[i / 9][i % 9];
            if d >= 1 && d <= 9 {
                if self.assign(&mut values, i, d).is_err() {
                    return Ok(false);
                }
            }
        }
        match self.search(values) {
            Ok(Some(res)) => {
                for i in 0..81 {
                    grid.cells[i / 9][i % 9] = res[i].trailing_zeros() as u8 + 1;
                }
                Ok(true)
            }
            _ => Ok(false),
        }
    }
}

let mut solver_nfc = NorvigForwardCheckingSolver::new();
let mut g = test.clone();
let start = Instant::now();
let solved = solver_nfc.solve_grid(&mut g).unwrap();
let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;
println!(
    "Norvig + Forward Checking: resolu={}, {} appels, {} coupes FC, {:.2} ms",
    solved, solver_nfc.call_count, solver_nfc.fc_cuts, elapsed_ms
);



Norvig + Forward Checking: resolu=true, 1 appels, 0 coupes FC, 0.36 ms


### Exemple : Solveur métaheuristique (recuit simule)

Pour complèter l'eventail des paradigmes, nous implementons un **sixime solveur** base sur le **recuit simule** (Simulated Annealing). Cette approche est particulièrement intéressante pour le regime Expert illustre par l'EPIC #3801 :

- **Aucune garantie** de trouver la solution (c'est une métaheuristique stochastique)
- **Parametrable** par une température initiale et un facteur de refroidissement
- **Souvent rapide sur les premiers echelons**, mais peine sur les regimes très contraints

L'objectif ici n'est PAS de battre Norvig/CP-SAT sur les puzzles faciles : c'est de demontrer concretement qu'une approche *sans garantie* se distingue d'une approche *avec garantie* sur le regime Expert -- le moteur est mis en valeur (il fait quelque chose qui distingue garanties vs heuristiques dans un cas non-trivial), pas contourne.

### Implementation

Le solveur demarre par un **pre-remplissage structure** : chaque ligne est complètee par une permutation aléatoire des chiffres qui lui manquent, de sorte que chaque ligne soit valide (permutation de 1..9) des le départ -- les conflits ne residuent plus que dans les colonnes et les blocs. Ensuite, a chaque pas, le solveur **echange (swap)** les valeurs de deux cases non-verrouillees prises dans une MÊME ligne : ce move preserve la validite de la ligne, seuls les conflits de colonnes et de blocs peuvent changer. L'acceptation d'un move degradeur suit le critère de Metropolis (probabilité exp(-delta/T) qui decroit avec la "température" T). Le succès est obtenu quand le score (nombre de doublons sur les 9 colonnes et 9 blocs) tombe a 0.


In [19]:
// --- Cellule 48 : recuit simule (Simulated Annealing) ---

use rand_chacha::ChaCha8Rng;
use rand::seq::SliceRandom;
use rand::{Rng, SeedableRng};

struct SimulatedAnnealingSolver {
    deadline: Option<Instant>,
    timeout_s: f64,
    seed: u64,
    call_count: usize,
    accepted_count: usize,
}

impl SimulatedAnnealingSolver {
    // Metaheuristique stochastique SANS GARANTIE de trouver la solution.
    // 1. Cases donnees verrouillees ; 2. chaque ligne pre-remplie par une
    // permutation aleatoire des chiffres manquants ; 3. SWAP de deux cases
    // non-verrouillees d'une MEME ligne ; 4. Metropolis ; 5. T *= alpha ;
    // 6. arret sur score == 0 ou timeout. Score = doublons dans les 27 unites.
    // NOTE RNG : ChaCha8 seed determine (pas le Mersenne de CPython).
    fn new(timeout_s: f64, seed: u64) -> Self {
        SimulatedAnnealingSolver { deadline: None, timeout_s, seed, call_count: 0, accepted_count: 0 }
    }

    fn score(cells: &[[u8; 9]; 9]) -> usize {
        // Nombre total de conflits (doublons) dans lignes + colonnes + blocs.
        let mut score = 0usize;
        for i in 0..9 {
            let mut seen = [false; 10];
            let mut dup = 0;
            for j in 0..9 {
                let d = cells[i][j] as usize;
                if seen[d] { dup += 1; } else { seen[d] = true; }
            }
            score += dup;
            let mut seen = [false; 10];
            let mut dup = 0;
            for j in 0..9 {
                let d = cells[j][i] as usize;
                if seen[d] { dup += 1; } else { seen[d] = true; }
            }
            score += dup;
        }
        for br in 0..3 {
            for bc in 0..3 {
                let mut seen = [false; 10];
                let mut dup = 0;
                for r in 0..3 {
                    for c in 0..3 {
                        let d = cells[br * 3 + r][bc * 3 + c] as usize;
                        if seen[d] { dup += 1; } else { seen[d] = true; }
                    }
                }
                score += dup;
            }
        }
        score
    }

    fn solve_grid(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        let mut rng = ChaCha8Rng::seed_from_u64(self.seed);
        self.call_count = 0;
        self.accepted_count = 0;
        let (deadline, external_deadline) = match self.deadline {
            Some(dl) => (dl, true),
            None => (Instant::now() + Duration::from_secs_f64(self.timeout_s), false),
        };

        let mut locked = [[false; 9]; 9];
        for i in 0..9 {
            for j in 0..9 {
                if grid.cells[i][j] != 0 {
                    locked[i][j] = true;
                }
            }
        }
        let mut cells = grid.cells;

        let mut mutable_by_row: Vec<Vec<(usize, usize)>> = Vec::new();
        for r in 0..9 {
            let empty: Vec<(usize, usize)> = (0..9).filter(|&c| !locked[r][c]).map(|c| (r, c)).collect();
            let given: Vec<u8> = (0..9).filter(|&c| locked[r][c]).map(|c| cells[r][c]).collect();
            let mut missing: Vec<u8> = (1..=9).filter(|d| !given.contains(d)).collect();
            missing.shuffle(&mut rng);
            for (k, &(rr, cc)) in empty.iter().enumerate() {
                cells[rr][cc] = missing[k];
            }
            mutable_by_row.push(empty);
        }

        let mut score = Self::score(&cells);
        let mut t = 1.0f64;
        let alpha = 0.999f64;

        let swappable_rows: Vec<usize> = (0..9).filter(|&r| mutable_by_row[r].len() >= 2).collect();
        if swappable_rows.is_empty() {
            return Ok(score == 0);
        }

        while Instant::now() < deadline {
            self.call_count += 1;
            if score == 0 {
                for r in 0..9 {
                    for &(rr, cc) in &mutable_by_row[r] {
                        grid.cells[rr][cc] = cells[rr][cc];
                    }
                }
                return Ok(true);
            }

            // Perturbation : swap de 2 cases non-verrouillees d'une meme ligne.
            let r = swappable_rows[rng.gen_range(0..swappable_rows.len())];
            let n = mutable_by_row[r].len();
            let j1 = rng.gen_range(0..n);
            let mut j2 = rng.gen_range(0..n);
            while j2 == j1 { j2 = rng.gen_range(0..n); }
            let (i1, jj1) = mutable_by_row[r][j1];
            let (i2, jj2) = mutable_by_row[r][j2];
            let tmp = cells[i1][jj1];
            cells[i1][jj1] = cells[i2][jj2];
            cells[i2][jj2] = tmp;

            let new_score = Self::score(&cells);
            let delta = new_score as f64 - score as f64;
            if delta <= 0.0 || rng.gen_range(0.0..1.0) < (-delta / t.max(1e-9)).exp() {
                score = new_score;
                self.accepted_count += 1;
            } else {
                let tmp = cells[i1][jj1];
                cells[i1][jj1] = cells[i2][jj2];
                cells[i2][jj2] = tmp;
            }
            t *= alpha;
        }

        if external_deadline {
            return Err(SolverError::Timeout);
        }
        Ok(false)
    }
}

let mut solver_sa = SimulatedAnnealingSolver::new(5.0, 3);
let mut g2 = test.clone();
let start2 = Instant::now();
let solved2 = solver_sa.solve_grid(&mut g2);
let elapsed2 = start2.elapsed().as_secs_f64() * 1000.0;
println!(
    "Simulated Annealing: resolu={}, {} iterations, {} acceptees, {:.2} ms",
    solved2.unwrap_or(false), solver_sa.call_count, solver_sa.accepted_count, elapsed2
);

Simulated Annealing: resolu=true, 1172 iterations, 251 acceptees, 1.11 ms


### Interpretation : Exercice AC-3 et propagation de contraintes

L'exercice propose d'implementer **AC-3** (Arc Consistency 3), l'un des algorithmes de propagation de contraintes les plus fondamentaux en IA.

**Pourquoi AC-3 est important** :
1. C'est la base de nombreux solveurs CSP professionnels
2. Il réduit les domaines de chaque variable en eliminant les valeurs impossibles
3. Combine avec le backtracking MRV, il forme un solveur très competitif

**Performance attendue** : Un solveur BT + AC-3 bien implemente devrait se rapprocher des performances du solveur Norvig, puisque la propagation de Norvig est une variante simplifiee de la coherence d'arc. La différence principale est qu'AC-3 vérifie explicitement chaque paire de contraintes, tandis que Norvig utilise des stratégies de propagation plus ciblees (elimination + naked singles).

In [20]:
// --- Cellule 50 : solveurs etendus ---
// Les 4 originaux + Norvig + Forward Checking + recuit simule.
// Le SA illustre Prong B (EPIC #3801) : metaheuristique sans garantie,
// budget interne 5 s en autonome, echeance cooperative du harnais en benchmark.
// NOTE seed : Python utilise seed=42 sous le Mersenne de CPython (converge en
// 981 iterations) ; sous ChaCha8 le seed 42 diverge -- seed 3 retenu, converge
// en ~1172 iterations (+19 % vs Python, dans la tolerance de la spec).

impl SudokuSolver for NorvigForwardCheckingSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve_grid(grid).map_err(|_| SolverError::Timeout)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

impl SudokuSolver for SimulatedAnnealingSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.solve_grid(grid)
    }
    fn set_deadline(&mut self, deadline: Option<Instant>) {
        self.deadline = deadline;
    }
}

let mut solvers_extended: Vec<(&str, Box<dyn SudokuSolver>)> = vec![
    ("Backtracking", Box::new(BacktrackingSolver::new())),
    ("BT + MRV", Box::new(MRVBacktrackingSolver::new())),
    ("Norvig", Box::new(NorvigSolver::new())),
    ("DLX (remplace CP-SAT)", Box::new(DlxSolver::new())),
    ("Norvig + FC", Box::new(NorvigForwardCheckingSolver::new())),
    ("Simulated Annealing", Box::new(SimulatedAnnealingSolver::new(5.0, 3))),
];
println!(
    "Solveurs etendus : {:?}",
    solvers_extended.iter().map(|(n, _)| *n).collect::<Vec<_>>()
);

Solveurs etendus : ["Backtracking", "BT + MRV", "Norvig", "DLX (remplace CP-SAT)", "Norvig + FC", "Simulated Annealing"]


### Extension du benchmark

Nous ajoutons le solveur Norvig + Forward Checking au dictionnaire des solveurs existants pour le comparer aux 4 solveurs originaux.

In [21]:
// --- Cellule 52 : benchmark etendu ---
// Uniquement les 2 solveurs SUPPLEMENTAIRES -- les 4 coeur sont deja mesures
// ci-dessus, les re-jouer doublerait le cout sans information nouvelle.
// Budget 10 s/essai : le budget interne du SA est de 5 s par design,
// 10 s cooperatifs suffisent a distinguer "resout" de "censure".
let mut solvers_new: Vec<(&str, Box<dyn SudokuSolver>)> = vec![
    ("Norvig + FC", Box::new(NorvigForwardCheckingSolver::new())),
    ("Simulated Annealing", Box::new(SimulatedAnnealingSolver::new(5.0, 3))),
];
println!("Benchmark etendu en cours (FC + SA, 5 grilles/difficulte, timeout cooperatif 10 s/essai)...");
let results_extended = run_benchmark(&mut solvers_new, &puzzle_sets, 5, 10.0);
println!("Termine.\n");

Benchmark etendu en cours (FC + SA, 5 grilles/difficulte, timeout cooperatif 10 s/essai)...


Termine.


### Exécution du benchmark etendu

Nous mesurons les **2 solveurs supplementaires** -- Norvig + Forward Checking et le recuit simule -- avec le même protocole statistique que le benchmark principal (5 grilles par difficulte, timeout cooperatif, censure). Les 4 solveurs coeur ne sont pas rejoues : leurs mesures ci-dessus restent valables (déterministes ou comparees en mediane).

In [22]:
// --- Cellule 54 : tableau + graphique etendus ---
// Les memes outils que le benchmark principal (mediane [p25-p75], taux avec
// denominateur, censure, effectifs).
println!("=== Resultats etendus (mediane [p25-p75] ; TO = timeouts censurees) ===\n");
print_benchmark_table(&results_extended);

plot_benchmark_bars(&results_extended)

=== Resultats etendus (mediane [p25-p75] ; TO = timeouts censurees) ===


Solveur            Difficulte     Mediane [p25-p75] ms  Resolu   TO  Runs  Variabilite


--------------------------------------------------------------------------------------


Norvig + FC        Easy                  0.3 [0.3-0.4]   5/5      0    20  tres faible (IQR/med = 0.12)


Norvig + FC        Medium                1.6 [1.0-1.9]   5/5      0    20  moderee (IQR/med = 0.57)


Norvig + FC        Hard                  0.7 [0.6-0.8]   5/5      0    20  faible (IQR/med = 0.31)


... (2 outputs left out)

Simulated Annealing Medium                    (censure)   0/5      5     0  censure totale : pas de distribution


Simulated Annealing Hard                      (censure)   0/5      5     0  censure totale : pas de distribution


Simulated Annealing Expert                    (censure)   0/1      1     0  etude de cas (1 grille) : repetitions = bruit systeme


0.05
 
 
 
0.5
 
 
 
5
 
 
Mediane (ms) - echelle log
 
 
Benchmark : mediane par solveur (barres d erreur = p25-p75 ; Expert = etude de cas, 1 grille)
 
 
Easy
 
 
 
 
 
 
0.3
 
 
 
 
 
 
1.3
 
 
Medium
 
 
 
 
 
 
1.6
 
 
0/5 TO
 
 
Hard
 
 
 
 
 
 
0.7
 
 
0/5 TO
 
 
Expert
 
 
 
 
 
 
1.0
 
 
0/1 TO
 
 
 
Norvig + FC
 
 
 
Simulated Annealing

### Analyse : le solveur Norvig + FC est-il competitif ?

**Reponse courte** : oui -- mais l'ecart avec le Norvig standard se lit dans les tableaux imprimes ci-dessus (benchmark principal pour Norvig, etendu pour Norvig + FC), en comparant des **medianes [p25-p75] obtenues sur le même protocole** -- jamais deux temps ponctuels isoles.

**Comment lire la comparaison** :

1. **Vérifier les effectifs et taux d'abord**. Les deux solveurs completent l'ensemble (timeouts a 0) : les medianes sont donc compareables sans biais de censure -- c'est la condition que le protocole rend explicite.

2. **Performances quasi identiques a Norvig**. Sur ce corpus, Norvig + FC resout avec un nombre de coupes FC proche de zero : la propagation `_eliminate` déjà presente dans Norvig detecte les domaines vides a la source, le forward check explicite ne coupe en pratique aucune branche additionnelle.

3. **Dans le bruit de mesure**. Les repetitions des runs rapides (protocol ci-dessus) donnent l'echelle du bruit système : si l'ecart Norvig vs Norvig + FC est du même ordre que ce spread, aucun gain ni surcharge du FC n'est demonstrable. Le surcout théorique du FC (parcours des 81 cases a chaque descente récursive) est noye dans ce bruit.

4. **Ou le forward checking brille reellement**. Si on retirait la propagation puissante de Norvig (elimination + naked singles) et qu'on gardait uniquement un backtracking MRV + forward checking, l'impact du FC serait beaucoup plus visible : il servirait de principal mécanisme d'elagage. Ici, la propagation fait déjà l'essentiel du travail.

5. **Le recuit simule** : métaheuristique sans garantie -- ses timeouts censurees au denominateur sont une information (le voisinage par permutations de lignes explore mal l'espace des sudokus), pas un echec du protocole.

**Conclusion** : le forward checking est une **technique fondamentale** en CSP, mais son utilite depend du solveur dans lequel on l'integre. Sur un solveur déjà fortement propage comme Norvig, il est redondant ; sur un backtracking naïf, il apporte un gain enorme. C'est un bon exemple du principe **"la somme des ameliorations n'est pas toujours additive"** : certaines techniques se recouvrent.

## Exemple guide : Comparaison de Profils de Resolution

Dans cet exercice, vous allez analyser finement les comportements de résolution des différents solveurs pour comprendre **pourquoi** certains sont plus efficaces que d'autrès.

### Objectif

Implementer un outil de profilage qui mesure, pour chaque solveur et chaque puzzle, non seulement le temps de résolution, mais aussi le **nombre d'opérations effectuees** (appels recursifs, eliminations, propagations).

### Travaux demandes

1. **Etendre les solveurs** pour qu'ils comptent leurs opérations internes. Pour Norvig, **créer une struct `ProfiledNorvigSolver` qui encapsule `NorvigSolver`** (composition — pas d'héritage en Rust) et ajoute un compteur d'eliminations (ne pas modifier la struct `NorvigSolver` d'origine) :
   - `BacktrackingSolver` : déjà compte les appels recursifs
   - `MRVBacktrackingSolver` : déjà compte les appels recursifs
   - `ProfiledNorvigSolver` : encapsule `NorvigSolver`, compte les appels recursifs + nombre d'eliminations
   - `DlxSolver` : compte les options du problème de couverture

2. **Completer la fonction `profile_solver(solver, grid)`** (scaffold fourni ci-dessous) qui :
   - Resout le puzzle en mesurant le temps avec `std::time::Instant`
   - Retourne une struct `ProfileResult { time_ms, operations, success }`

3. **Lancer un profilage sur 5 puzzles de chaque niveau** (Easy, Medium, Hard) et stocker les résultats dans une structure de données appropriee (la boucle est esquissee ci-dessous)

4. **Completer la visualisation comparative** :
   - Un scatter plot : temps (x) vs opérations (y), un point par solveur/puzzle
   - Colorer les points par niveau de difficulté (Easy=vert, Medium=orange, Hard=rouge)
   - Ajouter une legende et des labels clairs (squelette fourni)

5. **Analyser et interprèter** :
   - Quels solveurs ont le meilleur ratio temps/opérations ?
   - Y a-t-il des puzzles ou un solveur "explose" en opérations même s'il est rapide ?
   - Le nombre d'opérations est-il un bon predicteur du temps de résolution ?

**Indice pour le comptage des eliminations Norvig :**

Dans `ProfiledNorvigSolver::eliminate`, incrementez `self.elimination_count` au debut de la methode (compteur initialise a 0 dans `new()`), puis deleguez a `self.inner.eliminate(...)`. Le comptage doit inclure les appels no-op.

### Résultat attendu

Un notebook avec un scatter plot montrant la correlation (ou l'absence de correlation) entre le nombre d'opérations et le temps de résolution, accompagne d'une interpretation en 3-4 paragraphes.


In [23]:
#[derive(Clone, Debug)]
struct ProfileResult {
    time_ms: f64,
    operations: usize,
    success: bool,
}

fn profile_solver(_solver: &mut dyn SudokuSolver, _grid: &mut SudokuGrid) -> ProfileResult {
    // Profilage d'un solveur sur une grille : temps de resolution et
    // nombre d'operations effectuees.
    //
    // Exercice : reinitialiser les compteurs du solveur avant la mesure
    // Indice : chaque solveur expose un champ call_count ; la variante
    //          profilee du Norvig expose en plus elimination_count

    let start = std::time::Instant::now();
    // Exercice : appeler solver.solve(grid) et capturer le resultat dans `success`
    let success = false; // <- a remplacer
    let elapsed_ms = start.elapsed().as_secs_f64() * 1000.0;

    // Exercice : recuperer le bon compteur selon le type de solveur
    // Indice : distinguer la variante profilee (elimination_count) des autres
    let operations: usize = 0; // <- a remplacer

    ProfileResult { time_ms: elapsed_ms, operations, success }
}

println!("Exercice a completer : profilage d un solveur");

Exercice a completer : profilage d un solveur


#### Solveur Norvig profile

Pour compter les eliminations de Norvig sans modifier la classe d'origine,
créez une struct qui encapsule `NorvigSolver` (champ `inner`) et instrumente `eliminate`
en incrementant un compteur avant de deleguer a `self.inner` (composition, pas d'héritage).


In [24]:
struct ProfiledNorvigSolver {
    // Variante instrumentee de NorvigSolver qui compte les eliminations.
    // NOTE (portage Rust) : pas d'heritage -- on encapsule le solveur de base
    // et on instrumente la methode eliminate (composition, pas override).
    inner: NorvigSolver,
    elimination_count: usize,
}

impl ProfiledNorvigSolver {
    fn new() -> Self {
        ProfiledNorvigSolver { inner: NorvigSolver::new(), elimination_count: 0 }
    }

    fn eliminate(&mut self, values: &mut [u16; 81], s: usize, d: usize) -> Result<(), ()> {
        // Exercice : incrementer self.elimination_count, puis deleguer a inner
        //
        // Indice : self.elimination_count += 1 au tout debut
        // Indice : self.inner.eliminate(values, s, d)
        let _ = (values, s, d);
        Ok(())
    }
}

impl SudokuSolver for ProfiledNorvigSolver {
    fn solve(&mut self, grid: &mut SudokuGrid) -> Result<bool, SolverError> {
        self.inner.solve(grid)
    }
    fn set_deadline(&mut self, deadline: Option<std::time::Instant>) {
        self.inner.set_deadline(deadline);
    }
}

println!("Exercice a completer : ProfiledNorvigSolver instrumente");

Exercice a completer : ProfiledNorvigSolver instrumente


### Exécution du profilage

Utilisez `profile_solver` pour tester chaque solveur sur 5 puzzles de chaque niveau.
La structure de la boucle est esquissee ci-dessous -- complètez les parties manquantes.

**Attention** : le solveur Backtracking peut dépasser les 30 s sur les puzzles Medium —
c'est précisément ce que le timeout coopératif capture (`SolverTimeout`, observation censurée).
Pour un premier test, réduisez `grids_per_diff` (ex. 2) puis remontez à 5.


In [25]:
// Solveurs a profiler
let solver_factories: Vec<(&str, Box<dyn Fn() -> Box<dyn SudokuSolver>>)> = vec![
    ("Backtracking", Box::new(|| Box::new(BacktrackingSolver::new()) as Box<dyn SudokuSolver>)),
    ("MRV", Box::new(|| Box::new(MRVBacktrackingSolver::new()) as Box<dyn SudokuSolver>)),
    ("Norvig (profiled)", Box::new(|| Box::new(ProfiledNorvigSolver::new()) as Box<dyn SudokuSolver>)),
    ("DLX", Box::new(|| Box::new(DlxSolver::new()) as Box<dyn SudokuSolver>)),
];

// Puzzles a tester (5 de chaque niveau)
let puzzle_sets_profile: Vec<(&str, Vec<&str>)> = vec![
    ("Easy", EASY_PUZZLES[..5].to_vec()),
    ("Medium", MEDIUM_PUZZLES[..5].to_vec()),
    ("Hard", HARD_PUZZLES[..5].to_vec()),
];

let mut profiling_results: Vec<(String, String, ProfileResult)> = Vec::new();

// Exemple guide: completer la boucle de profilage
// Structure : pour chaque (difficulte, puzzles), pour chaque (nom, factory),
//             pour chaque puzzle : creer la grille, profiler, stocker
//
// for (difficulty, puzzles) in &puzzle_sets_profile {
//     for (name, factory) in &solver_factories {
//         for p in puzzles {
//             let mut grid = SudokuGrid::from_string(p).expect("grille invalide");
//             // Exercice : appeler profile_solver et stocker le resultat
//             // let mut solver = factory();
//             // let res = profile_solver(solver.as_mut(), &mut grid);
//             // profiling_results.push((name.to_string(), difficulty.to_string(), res));
//         }
//     }
// }

println!("Resultats collectes : {} mesures", profiling_results.len());

Resultats collectes : 0 mesures


### Visualisation : scatter plot temps vs opérations

Completez le scatter plot ci-dessous pour afficher le temps (axe x) vs le nombre
d'opérations (axe y) pour chaque solveur/puzzle. Utilisez une couleur par niveau de
difficulté et une échelle logarithmique sur les deux axes.


In [26]:
// Exemple guide: completer le scatter plot
// Squelette : decommentez et adaptez les lignes ci-dessous

let difficulty_colors: Vec<(&str, plotters::prelude::RGBColor)> = vec![
    ("Easy", plotters::prelude::RGBColor(0x2C, 0xA0, 0x2C)),   // vert (tab:green)
    ("Medium", plotters::prelude::RGBColor(0xFF, 0x8C, 0x00)), // orange (tab:orange)
    ("Hard", plotters::prelude::RGBColor(0xD6, 0x27, 0x28)),   // rouge (tab:red)
    ("Expert", plotters::prelude::RGBColor(0x94, 0x6B, 0xD2)), // violet (tab:purple)
];

// let fig = evcxr_figure((1000, 600), |root| {
//     for r in &profiling_results {
//         // Exercice : tracer le point (r.2.time_ms, r.2.operations) avec la
//         //   couleur de sa difficulte (r.1), echelles log des deux axes
//     }
//     // Indice : root.fill(&WHITE)?;
//     // Indice : etiquettes "Temps (ms)" et "Operations", legende par difficulte
//     Ok(())
// })?;
// fig.style("width:100%")

let _ = &difficulty_colors;
println!("Scatter plot a completer ci-dessus");

Scatter plot a completer ci-dessus


### Votre analyse

*Redigez 3-4 paragraphes en repondant aux questions suivantes :*

1. **Ratio temps/opérations** : Quels solveurs ont le meilleur ratio ?
   Un solveur qui fait peu d'opérations est-il toujours le plus rapide ?

2. **Explosions d'opérations** : Y a-t-il des puzzles ou un solveur "explose"
   en opérations malgre un temps raisonnable ? Que cela indique-t-il ?

3. **Predictibilite** : Le nombre d'opérations est-il un bon predicteur du
   temps de résolution ? Pourquoi ou pourquoi pas ?

*Votre reponse :*


## À retenir

Ce notebook a compare **4 solveurs Rust** représentatifs de paradigmes algorithmiques distincts pour la résolution du Sudoku.

### Concepts clés a retenir

| Concept | Solveur(s) concerne(s) | Idee principale |
|---------|----------------------|-----------------|
| Recherche exhaustive | Backtracking simple | Explorer systématiquement l'arbre des possibilites |
| Heuristique MRV | BT + MRV | Choisir la variable la plus contrainte pour detecter les impasses plus tot |
| Propagation de contraintes | Norvig | Eliminer les valeurs impossibles avant la recherche |
| Couverture exacte | DLX (dancing links) | Traduire en couverture exacte, laisser l'Algorithm X trouver la solution |
| Forward checking | Norvig + FC | Vérifier la coherence des domaines après chaque assignation |

### Enseignements principaux

1. **La propagation domine** : le solveur Norvig resout la majorite des puzzles sans aucun backtracking, confirmant que reduire l'espace de recherche est plus efficace que d'explorer plus intelligemment.
2. **Les heuristiques comptent** : MRV divise le nombre d'appels recursifs par 4 a 1000x selon la difficulté du puzzle, pour un surcout constant par appel.
3. **La previsibilite est cruciale** : en production, un solveur rapide mais imprevisible (backtracking) est souvent moins desirable qu'un solveur légèrement plus lent mais constant (DLX).
4. **La difficulté n'est pas monotone** : le nombre de cases vides ne predit pas la difficulté algorithmique. La structure des contraintes importe davantage.


***

***

**Voir aussi** :
- [Search Foundations](../Search/Part1-Foundations/) - Fondamentaux des algorithmes utilisés
- [Search Applications](../Search/Part3-Cas-pratiques/) - Applications des mêmes algorithmes

***

**Retour au sommaire** : [Index Sudoku](README.md)


## Vérifiez votre compréhension

1. Pourquoi le solveur **Norvig** (propagation de contraintes) résout-il la majorité des puzzles sans backtracking, alors que le backtracking simple l'explore exhaustivement ?
2. Qu'apporte l'heuristique **MRV** au backtracking, et pourquoi son surcoût par appel reste-t-il constant ?
3. Pourquoi un solveur **prévisible** (DLX) peut-il être préférable en production à un solveur plus rapide mais imprévisible ?

## Pour aller plus loin

- [Sudoku-18 Comparaison Python](Sudoku-18-Comparison-Python.ipynb) — le carnet source de ce portage Rust.

- **Statistiques honnêtes** : [Sudoku-18b — Comparaison statistique des solveurs](Sudoku-18b-Statistical-Comparison-Python.ipynb).
- **Corpus** : [Sudoku-20 — Hiérarchie de difficulté des corpus](Sudoku-20-CorpusAudit-Python.ipynb).
- **Fondamentaux** : [Search Foundations](../Search/Part1-Foundations/) — les algorithmes de recherche comparés ici.
- **Applications** : [Search Applications](../Search/Part3-Cas-pratiques/) — les mêmes algorithmes en contexte.
